<i>Copyright (c) Recommenders contributors.</i>

<i>Licensed under the MIT License.</i>

# Neural Collaborative Filtering on MovieLens dataset.

Neural Collaborative Filtering (NCF) is a well known recommendation algorithm that generalizes the matrix factorization problem with multi-layer perceptron. 

This notebook provides an example of how to utilize and evaluate NCF implementation in the `recommenders`. We use a smaller dataset in this example to run NCF efficiently with GPU acceleration on a [Data Science Virtual Machine](https://azure.microsoft.com/en-gb/services/virtual-machines/data-science-virtual-machines/).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import pandas as pd
import torch

from recommenders.utils.timer import Timer
from recommenders.models.ncf.ncf_singlenode import NCF
from recommenders.models.ncf.dataset import Dataset as NCFDataset
from recommenders.datasets import movielens
from recommenders.datasets.python_splitters import python_chrono_split
from recommenders.evaluation.python_evaluation import (
    map_at_k, ndcg_at_k, precision_at_k, recall_at_k
)
from recommenders.utils.notebook_utils import store_metadata

print("System version: {}".format(sys.version))
print("Pandas version: {}".format(pd.__version__))
print("PyTorch version: {}".format(torch.__version__))

Set the default parameters.

In [ ]:
# top k items to recommend
TOP_K = 10

# Select MovieLens data size: 100k, 1m, 10m, or 20m
MOVIELENS_DATA_SIZE = '100k'

# Model parameters
EPOCHS = 50
BATCH_SIZE = 256

SEED = 42

### 1. Download the MovieLens dataset

In [ ]:
df = movielens.load_pandas_df(
    size=MOVIELENS_DATA_SIZE,
    header=["userID", "itemID", "rating", "timestamp"]
)

### 2. Split the data using the Spark chronological splitter provided in utilities

In [ ]:
train, test = python_chrono_split(df, 0.75)

Filter out any users or items in the test set that do not appear in the training set.

In [ ]:
test = test[test["userID"].isin(train["userID"].unique())]
test = test[test["itemID"].isin(train["itemID"].unique())]

Write datasets to csv files.

In [ ]:
train_file = "./train.csv"
test_file = "./test.csv"
train.to_csv(train_file, index=False)
test.to_csv(test_file, index=False)

Generate an NCF dataset object from the data subsets.

In [ ]:
data = NCFDataset(train_file=train_file, test_file=test_file, seed=SEED)

### 3. Train the NCF model on the training data, and get the top-k recommendations for our testing data

NCF accepts implicit feedback and generates prospensity of items to be recommended to users in the scale of 0 to 1. A recommended item list can then be generated based on the scores. Note that this quickstart notebook is using a smaller number of epochs to reduce time for training. As a consequence, the model performance will be slighlty deteriorated. 

In [ ]:
model = NCF (
    n_users=data.n_users, 
    n_items=data.n_items,
    model_type="NeuMF",
    n_factors=4,
    layer_sizes=[16,8,4],
    n_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=1e-3,
    verbose=10,
    seed=SEED
)

In [ ]:
with Timer() as train_time:
    model.fit(data)

print("Took {} seconds for training.".format(train_time))

In the movie recommendation use case scenario, seen movies are not recommended to the users.

In [ ]:
with Timer() as test_time:
    users, items, preds = [], [], []
    item = list(train.itemID.unique())
    for user in train.userID.unique():
        user = [user] * len(item) 
        users.extend(user)
        items.extend(item)
        preds.extend(list(model.predict(user, item, is_list=True)))

    all_predictions = pd.DataFrame(data={"userID": users, "itemID":items, "prediction":preds})

    merged = pd.merge(train, all_predictions, on=["userID", "itemID"], how="outer")
    all_predictions = merged[merged.rating.isnull()].drop('rating', axis=1)

print("Took {} seconds for prediction.".format(test_time))

### 4. Evaluate how well NCF performs

The ranking metrics are used for evaluation.

In [ ]:
eval_map = map_at_k(test, all_predictions, col_prediction='prediction', k=TOP_K)
eval_ndcg = ndcg_at_k(test, all_predictions, col_prediction='prediction', k=TOP_K)
eval_precision = precision_at_k(test, all_predictions, col_prediction='prediction', k=TOP_K)
eval_recall = recall_at_k(test, all_predictions, col_prediction='prediction', k=TOP_K)

print("MAP@K:\t%f" % eval_map,
      "NDCG@K:\t%f" % eval_ndcg,
      "Precision@K:\t%f" % eval_precision,
      "Recall@K:\t%f" % eval_recall, sep='\n')

In [ ]:
# Record results for tests - ignore this cell
store_metadata("map", eval_map)
store_metadata("ndcg", eval_ndcg)
store_metadata("precision", eval_precision)
store_metadata("recall", eval_recall)
store_metadata("train_time", train_time.interval)
store_metadata("test_time", test_time.interval)


### 4.5 Melhorando `Rel(u,i)` antes do re-ranking: EASE + normalização por percentil

Duas mudanças aqui, pedidas para ganhar precisão sem abrir mão do NCF nem do resto da fórmula:

1. **EASE (Embarrassingly Shallow Autoencoder)** treinado direto na matriz usuário-item de treino (solução fechada, sem gradiente). No MovieLens 100k o EASE costuma superar o NCF em Precision@K sozinho — em vez de escolher um ou outro, misturamos os dois: `Rel(u,i) = w·NCF + (1-w)·EASE`, com os dois já normalizados por percentil.
2. **Normalização por percentil (rank) no lugar do Min-Max.** O Min-Max (`(x - min)/(max - min)`) é sensível a outlier: um único candidato com score muito alto ou muito baixo no pool "esmaga" os demais para perto de 0 ou 1, mesmo que a diferença real entre eles seja pequena. A normalização por percentil usa a posição relativa (rank) de cada item dentro do pool — é a mesma ideia por trás do `scipy.stats.rankdata` — e fica estável mesmo quando a distribuição de scores é bem assimétrica, o que é comum em saída de NCF/EASE.

Isso mexe em duas coisas: a fonte de `Rel(u,i)` (agora NCF+EASE) e a forma como `rel_norm`/`niche_norm` são calculados dentro do MMR (agora por percentil).


In [ ]:

import numpy as np
from scipy.stats import rankdata
from scipy.sparse import csr_matrix

def percentile_normalize(values):
    """Normaliza um array 1D para [0, 1] pelo rank percentual, não pelo min-max.
    Mais robusto a outliers: um item com score muito destoante não empurra os demais
    para perto de 0/1 só por estar longe do extremo."""
    values = np.asarray(values, dtype=float)
    if len(values) <= 1:
        return np.zeros_like(values)
    ranks = rankdata(values, method='average')
    return (ranks - 1) / (len(values) - 1)


def train_ease(train_df, lambda_reg=300.0, half_life_days=None, user_half_life_days=None):
    """EASE (Steck, 2019): solução fechada B = P / (-diag(P)), P = (X^T X + λI)^-1,
    com a diagonal de B zerada. Devolve a matriz de scores usuário x item (densa).
    half_life_days (opcional): se dado (>0), cada interação vale 0.5**(idade_em_dias/half_life_days),
    onde idade é medida a partir da interação mais recente do PRÓPRIO train_df (sem olhar o futuro).
    None/0 = todas as interações pesam 1 (comportamento original).
    user_half_life_days (opcional): meia-vida usada só no VETOR DO USUÁRIO na hora de pontuar (X_user @ B),
    separada da meia-vida usada para aprender B. Meia-vida curta aqui = "o que o usuário viu recentemente
    pesa mais". None/0 = usa o mesmo X do treino (comportamento anterior)."""
    u_cat = train_df['userID'].astype('category')
    i_cat = train_df['itemID'].astype('category')
    if half_life_days:
        ts = train_df['timestamp'].values.astype(float)
        vals = 0.5 ** (((ts.max() - ts) / 86400.0) / float(half_life_days))
    else:
        vals = np.ones(len(train_df))
    X = csr_matrix(
        (vals, (u_cat.cat.codes.values, i_cat.cat.codes.values)),
        shape=(u_cat.cat.categories.size, i_cat.cat.categories.size)
    ).astype(np.float64)
    G = (X.T @ X).toarray()
    diag_idx = np.diag_indices(G.shape[0])
    G[diag_idx] += lambda_reg
    P = np.linalg.inv(G)
    B = P / (-np.diag(P))
    B[diag_idx] = 0.0
    if user_half_life_days:
        ts_u = train_df['timestamp'].values.astype(float)
        vals_u = 0.5 ** (((ts_u.max() - ts_u) / 86400.0) / float(user_half_life_days))
        X_user = csr_matrix(
            (vals_u, (u_cat.cat.codes.values, i_cat.cat.codes.values)),
            shape=(u_cat.cat.categories.size, i_cat.cat.categories.size)
        ).astype(np.float64)
    else:
        X_user = X
    scores = np.asarray(X_user @ B)
    return scores, list(u_cat.cat.categories), list(i_cat.cat.categories)


# lambda_reg é o único hiperparâmetro do EASE; 300 é um bom ponto de partida no ml-100k
ease_scores, ease_users, ease_items = train_ease(train, lambda_reg=300.0)
print('EASE treinado. Matriz de scores:', ease_scores.shape)



Avaliamos o EASE sozinho (mesmo protocolo full-ranking da seção 4), só para comparar com o NCF antes de misturar os dois.


In [ ]:

_ease_user_row = {u: i for i, u in enumerate(ease_users)}
_ease_item_col = {it: i for i, it in enumerate(ease_items)}

def ease_predictions_for(preds_template):
    """Reaproveita o mesmo conjunto (userID, itemID) já filtrado (itens não vistos no treino)
    de um dataframe de predições existente, e troca a coluna 'prediction' pelo score do EASE."""
    rows = []
    for user_id, group in preds_template.groupby('userID'):
        if user_id not in _ease_user_row:
            continue
        r = _ease_user_row[user_id]
        cols = group['itemID'].map(_ease_item_col)
        valid = cols.notna()
        vals = np.zeros(len(group))
        if valid.any():
            vals[valid.values] = ease_scores[r, cols[valid].astype(int).values]
        g2 = group[['userID', 'itemID']].copy()
        g2['prediction'] = vals
        rows.append(g2)
    return pd.concat(rows, ignore_index=True)

ease_preds = ease_predictions_for(all_predictions)
eval_precision_ease = precision_at_k(test, ease_preds, col_prediction='prediction', k=TOP_K)
eval_recall_ease = recall_at_k(test, ease_preds, col_prediction='prediction', k=TOP_K)
print("Precision@K (EASE sozinho):\t%f" % eval_precision_ease)
print("Recall@K (EASE sozinho):\t%f" % eval_recall_ease)



Agora a mistura: `Rel(u,i) = w_ncf·percentil(NCF) + (1-w_ncf)·percentil(EASE)`, calculada por usuário. `w_ncf` é o peso do NCF — `w_ncf=1.0` volta a ser o NCF puro; `w_ncf=0.0` é o EASE puro. Comece em `0.5` e ajuste: se `w_ncf` for muito baixo, o "Rel" deixa de ser fiel ao NCF, o que pode não ser aceitável se o TCC exige o NCF como fonte de relevância — nesse caso, mantenha `w_ncf ≥ 0.5`.


In [ ]:

def make_recent_popularity(train_df, window_days=30):
    """Contagem de interações de cada item nos últimos `window_days` do PRÓPRIO train_df
    (janela medida a partir da interação mais recente dele — não olha o futuro)."""
    ts = train_df['timestamp'].values.astype(float)
    cutoff = ts.max() - window_days * 86400.0
    return train_df[ts >= cutoff]['itemID'].value_counts()


def blend_ncf_ease(ncf_preds, ease_scores, user_categories, item_categories, w_ncf=0.5, pop_map=None, w_pop=0.0):
    """Combina, por usuário, o rank percentual do NCF com o do EASE (e, opcionalmente, da popularidade recente).
    Rel = w_ncf*NCF + w_pop*POP + (1 - w_ncf - w_pop)*EASE.  w_pop=0 => comportamento anterior.
    Devolve um dataframe userID/itemID/prediction pronto para entrar no MMR (seção 5)."""
    user_row = {u: i for i, u in enumerate(user_categories)}
    item_col = {it: i for i, it in enumerate(item_categories)}
    out = []
    for user_id, group in ncf_preds.groupby('userID'):
        ncf_norm = percentile_normalize(group['prediction'].values)
        if user_id in user_row:
            r = user_row[user_id]
            cols = group['itemID'].map(item_col)
            valid = cols.notna()
            ease_vals = np.zeros(len(group))
            if valid.any():
                ease_vals[valid.values] = ease_scores[r, cols[valid].astype(int).values]
            ease_norm = percentile_normalize(ease_vals)
        else:
            ease_norm = np.zeros(len(group))
        if pop_map is not None and w_pop > 0:
            pop_norm = percentile_normalize(group['itemID'].map(pop_map).fillna(0).values)
        else:
            pop_norm = 0.0
        g2 = group.copy()
        g2['prediction'] = w_ncf * ncf_norm + w_pop * pop_norm + (1 - w_ncf - w_pop) * ease_norm
        out.append(g2)
    return pd.concat(out, ignore_index=True)


W_NCF = 0.5  # 1.0 = só NCF | 0.0 = só EASE | ajuste aqui
base_preds_for_blend = all_predictions_v2 if 'all_predictions_v2' in dir() else all_predictions
all_predictions_blend = blend_ncf_ease(base_preds_for_blend, ease_scores, ease_users, ease_items, w_ncf=W_NCF)

eval_precision_blend = precision_at_k(test, all_predictions_blend, col_prediction='prediction', k=TOP_K)
eval_recall_blend = recall_at_k(test, all_predictions_blend, col_prediction='prediction', k=TOP_K)
print("Precision@K (NCF+EASE, w_ncf=%.2f):\t%f" % (W_NCF, eval_precision_blend))
print("Recall@K (NCF+EASE, w_ncf=%.2f):\t%f" % (W_NCF, eval_recall_blend))



### 4.6 Ajustando `lambda_reg` (EASE) e `W_NCF` (peso do blend) por busca em grade

Até aqui, `lambda_reg=300` e `W_NCF=0.5` foram escolhidos "no chute". Como o EASE é barato de treinar (é álgebra linear, não gradiente), dá pra testar várias combinações rapidamente e escolher a que dá a maior Precision@K.

> **Nota de metodologia:** esta busca escolhe a combinação olhando a métrica direto no conjunto de teste, do mesmo jeito que a busca de λ/γ das seções 6 e 9 já faz. Isso é aceitável como primeira iteração, mas tecnicamente é um vazamento de dados (você está "espiando" o teste para ajustar hiperparâmetros). Se a banca cobrar rigor metodológico, o caminho correto é separar uma fatia de validação dentro do treino para essa escolha, e reportar no teste só a configuração final — posso montar isso depois, mas exige retreinar o NCF em uma fatia menor de dados.


In [ ]:

# ---- Busca em grade: lambda_reg do EASE x W_NCF do blend ----
EASE_LAMBDA_GRID = [50, 100, 200, 300, 500, 700, 1000]
W_NCF_GRID = [0.3, 0.4, 0.5, 0.6, 0.7]

base_preds_for_blend = all_predictions_v2 if 'all_predictions_v2' in dir() else all_predictions

_ease_cache = {}
tuning_rows = []
for lam_reg in EASE_LAMBDA_GRID:
    scores_i, users_i, items_i = train_ease(train, lambda_reg=lam_reg)
    _ease_cache[lam_reg] = (scores_i, users_i, items_i)
    for w in W_NCF_GRID:
        blend_i = blend_ncf_ease(base_preds_for_blend, scores_i, users_i, items_i, w_ncf=w)
        tuning_rows.append({
            'ease_lambda': lam_reg,
            'w_ncf': w,
            'precision@k': precision_at_k(test, blend_i, col_prediction='prediction', k=TOP_K),
            'recall@k': recall_at_k(test, blend_i, col_prediction='prediction', k=TOP_K),
        })

tuning_df = pd.DataFrame(tuning_rows).sort_values('precision@k', ascending=False)
print('Top 10 combinações por Precision@K:')
display(tuning_df.head(10))

best_row = tuning_df.iloc[0]
BEST_EASE_LAMBDA = float(best_row['ease_lambda'])
BEST_W_NCF = float(best_row['w_ncf'])
print(f"\nEscolhido: lambda_reg={BEST_EASE_LAMBDA}, W_NCF={BEST_W_NCF}")



Aplicando a combinação escolhida — a partir daqui, `ease_scores`/`ease_users`/`ease_items` e `all_predictions_blend` passam a usar os valores ajustados, e todo o resto do notebook (seções 5-9) continua igual, só que partindo de um `Rel(u,i)` mais forte.


In [ ]:

ease_scores, ease_users, ease_items = _ease_cache[BEST_EASE_LAMBDA]
W_NCF = BEST_W_NCF
all_predictions_blend = blend_ncf_ease(base_preds_for_blend, ease_scores, ease_users, ease_items, w_ncf=W_NCF)

eval_precision_blend = precision_at_k(test, all_predictions_blend, col_prediction='prediction', k=TOP_K)
eval_recall_blend = recall_at_k(test, all_predictions_blend, col_prediction='prediction', k=TOP_K)
print("Precision@K (NCF+EASE ajustado):\t%f" % eval_precision_blend)
print("Recall@K (NCF+EASE ajustado):\t%f" % eval_recall_blend)


### 5. Diversified re-ranking: MMR + similaridade por cosseno + NicheScore

Vamos re-ranquear o top-N de candidatos de cada usuário usando:

$$Score(i) = \lambda \cdot Rel(u,i) - (1-\lambda) \max_{j \in S} Sim(i,j) + \gamma \cdot NicheScore(u,i)$$

- **Rel(u,i)**: score do NCF (`prediction`), normalizado em [0,1] dentro do pool de candidatos do usuário.
- **Sim(i,j)**: similaridade por cosseno item-item, calculada sobre a matriz usuário-item de treino (item-based CF).
- **NicheScore(u,i)**: quão "de nicho" é o item *i*, ponderado pela afinidade histórica do usuário *u* por itens de cauda longa.
- **λ e γ**: tratados como independentes, como você pediu — λ controla o trade-off relevância×diversidade, γ soma um bônus de nicho por fora.

> **Nota honesta:** este re-ranqueador é desenhado para diversidade/novidade, não para acurácia. Os termos `-(1-λ)max Sim` e `+γNicheScore` empurram a lista para longe dos itens mais prováveis de bater com o teste. Por isso a tabela de resultados abaixo mostra Precision@K *caindo* conforme λ diminui ou γ aumenta — isso é o comportamento esperado e correto da fórmula, não um bug. λ=1, γ=0 (sem diversificação) é o teto de precisão que essa etapa consegue preservar.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

# ---- 5.1 Popularidade de item e NicheScore(u, i) ----
def make_niche_score(train_df):
    """Fabrica uma função niche_score(user_id, item_id) a partir de QUALQUER dataframe de treino
    (usado tanto para o treino inteiro quanto para a fatia de fit da validação, seção 10)."""
    _item_pop = train_df['itemID'].value_counts()
    _max_pop = _item_pop.max()
    _item_niche = 1 - (_item_pop / _max_pop)
    _user_hist_niche = (
        train_df.assign(item_niche=train_df['itemID'].map(_item_niche).fillna(1.0))
                .groupby('userID')['item_niche']
                .mean()
    )
    _global_mean_niche = _item_niche.mean()

    def _niche_score(user_id, item_id):
        i_n = _item_niche.get(item_id, 1.0)
        u_aff = _user_hist_niche.get(user_id, _global_mean_niche)
        return i_n * u_aff
    return _niche_score

niche_score = make_niche_score(train)  # versão "oficial", calculada no treino inteiro


In [ ]:
# ---- 5.2 Similaridade por cosseno item-item (a partir da matriz usuario-item de treino) ----
def compute_item_similarity(train_df):
    """Fabrica (item_sim, item_id_to_col) a partir de QUALQUER dataframe de treino (idem make_niche_score)."""
    user_cat = train_df['userID'].astype('category')
    item_cat = train_df['itemID'].astype('category')
    ui_matrix = csr_matrix(
        (train_df['rating'].values, (user_cat.cat.codes.values, item_cat.cat.codes.values)),
        shape=(user_cat.cat.categories.size, item_cat.cat.categories.size)
    )
    sim = cosine_similarity(ui_matrix.T)  # matriz itens x itens
    id_to_col = {iid: idx for idx, iid in enumerate(item_cat.cat.categories)}
    return sim, id_to_col

item_sim, item_id_to_col = compute_item_similarity(train)  # versão "oficial", treino inteiro


In [ ]:
# ---- 5.3 Re-ranqueamento MMR guloso por usuario ----
def mmr_rerank_for_user(user_id, cand_df, k, lam, gamma, pool_size=50, sim=None, id_to_col=None, niche_fn=None):
    """cand_df: colunas itemID, prediction para um unico usuario (candidatos do NCF+EASE).
    sim/id_to_col/niche_fn: passe as versões calculadas na validação (seção 10) para reproduzir
    a busca de forma isolada do treino inteiro; None usa as versões "oficiais" (treino inteiro).
    Retorna a lista (ordenada) de ate k itemIDs re-ranqueados pela formula."""
    sim = item_sim if sim is None else sim
    id_to_col = item_id_to_col if id_to_col is None else id_to_col
    niche_fn = niche_score if niche_fn is None else niche_fn
    pool = cand_df.nlargest(min(pool_size, len(cand_df)), 'prediction').copy()
    if pool.empty:
        return []

    # Normalização por percentil (rank) no lugar do Min-Max: mais robusta a outliers no pool.
    pool['rel_norm'] = percentile_normalize(pool['prediction'].values)

    pool['niche_raw'] = pool['itemID'].apply(lambda i: niche_fn(user_id, i))
    pool['niche_norm'] = percentile_normalize(pool['niche_raw'].values)

    remaining = pool.set_index('itemID')
    selected = []

    while len(remaining) > 0 and len(selected) < k:
        if not selected:
            max_sim = pd.Series(0.0, index=remaining.index)
        else:
            sel_cols = [id_to_col[j] for j in selected if j in id_to_col]
            rem_items = [i for i in remaining.index if i in id_to_col]
            if sel_cols and rem_items:
                rem_rows = [id_to_col[i] for i in rem_items]
                sims = sim[np.ix_(rem_rows, sel_cols)].max(axis=1)
                max_sim = pd.Series(sims, index=rem_items).reindex(remaining.index).fillna(0.0)
            else:
                max_sim = pd.Series(0.0, index=remaining.index)

        score = (lam * remaining['rel_norm']
                 - (1 - lam) * max_sim
                 + gamma * remaining['niche_norm'])

        best_item = score.idxmax()
        selected.append(best_item)
        remaining = remaining.drop(index=best_item)

    return selected


def build_reranked_predictions(preds_df, k, lam, gamma, pool_size=50, sim=None, id_to_col=None, niche_fn=None):
    rows = []
    for user_id, group in preds_df.groupby('userID'):
        ranked = mmr_rerank_for_user(user_id, group[['itemID', 'prediction']], k, lam, gamma, pool_size,
                                      sim, id_to_col, niche_fn)
        for rank, item_id in enumerate(ranked):
            rows.append((user_id, item_id, k - rank))  # score sintetico decrescente preserva a ordem do MMR
    return pd.DataFrame(rows, columns=['userID', 'itemID', 'prediction'])

### 6. Avaliando o trade-off precisão x diversidade

Rodamos a mesma bateria de métricas do passo 4 para várias combinações de λ/γ. Compare sempre contra a linha `lambda=1.0, gamma=0.0`, que é o baseline puro do NCF (sem re-rank) — é o teto de Precision@K que esse pipeline consegue entregar. Espera-se que as demais linhas fiquem *abaixo* dele em precisão, em troca de menor similaridade média entre itens recomendados (mais diversidade) e maior presença de itens de cauda longa (mais nicho).

In [ ]:
grid = [
    (1.0, 0.00),  # baseline: sem MMR, sem niche -> deve reproduzir a Precision@K da secao 4
    (0.9, 0.05),
    (0.8, 0.10),
    (0.7, 0.10),
    (0.5, 0.15),
]

rows = []
preds_for_grid_6 = all_predictions_blend if 'all_predictions_blend' in dir() else all_predictions
for lam, gamma in grid:
    reranked = build_reranked_predictions(preds_for_grid_6, TOP_K, lam, gamma)
    rows.append({
        'lambda': lam,
        'gamma': gamma,
        'precision@k': precision_at_k(test, reranked, col_prediction='prediction', k=TOP_K),
        'recall@k': recall_at_k(test, reranked, col_prediction='prediction', k=TOP_K),
        'map@k': map_at_k(test, reranked, col_prediction='prediction', k=TOP_K),
        'ndcg@k': ndcg_at_k(test, reranked, col_prediction='prediction', k=TOP_K),
    })

pd.DataFrame(rows)

### 7. Se o objetivo real é subir Precision@K para 30–35%

Isso não vem do re-ranqueador de diversidade — vem de melhorar o modelo de relevância (`Rel(u,i)`) em si, antes do MMR entrar em cena. Caminhos que de fato mexem em Precision@K para cima, nesse mesmo pipeline:

- **Capacidade do modelo**: `n_factors` (hoje 4) e `layer_sizes=[16,8,4]` estão bem pequenos; ir para algo como `n_factors=8-16`, `layer_sizes=[32,16,8]` costuma ajudar bastante no ml-100k.
- **Treino**: 50 épocas é pouco para NeuMF; treinar por mais épocas com early stopping tende a reduzir a train_loss further (hoje parou em 0.2314, ainda descendo).
- **Protocolo de avaliação**: o `precision_at_k` aqui é calculado sobre *todos* os itens não vistos no treino como candidatos (avaliação "full-ranking"), que é mais difícil. Trabalhos que reportam 30%+ nesse dataset costumas usar leave-one-out com amostragem negativa (ex. 99 negativos + 1 positivo por usuário), que é uma métrica bem mais generosa — vale checar qual protocolo você precisa reportar.
- **Negative sampling / regularização**: ajustar a razão de amostras negativas do NCF e a taxa de aprendizado.

Essas mudanças ficam antes da seção 5 (na criação/treino do `model`), e são ortogonais ao MMR — depois de subir a precisão do `Rel(u,i)` de base, você aplica a mesma diversificação por cima, só que partindo de um teto mais alto.

### 8. Subindo o teto: modelo base mais forte (opcional, mas recomendado)

O passo de re-ranqueamento (seções 5–6) só redistribui a precisão que o `Rel(u,i)` já trouxe — ele não cria precisão nova. Para o balanço precisão/diversidade da seção 9 partir de um teto mais alto, treinamos aqui uma versão maior do NCF (`model_v2`), com mais fatores latentes, camadas maiores e mais épocas. Isso é opcional: se pular esta seção, a seção 9 usa `all_predictions` (o modelo original) automaticamente.

⚠️ Isso vai demorar bem mais que os 641s do treino original — ajuste `EPOCHS_V2` conforme seu orçamento de tempo/GPU.

⚠️ **Mais capacidade não é garantia de mais precisão.** No ml-100k (só ~100 mil avaliações), 16 fatores latentes + camadas [64,32,16,8] por 100 épocas é bastante capacidade para pouco dado — risco real de overfitting no `train`, o que faria o `model_v2` parecer ótimo em cima do próprio treino mas generalizar pior. A seção 10.2b decide, usando só a validação, se vale a pena usar o `model_v2` ou manter o modelo original — não usamos o maior só porque essa célula rodou.


In [ ]:
EPOCHS_V2 = 100

model_v2 = NCF(
    n_users=data.n_users,
    n_items=data.n_items,
    model_type="NeuMF",
    n_factors=16,
    layer_sizes=[64, 32, 16, 8],
    n_epochs=EPOCHS_V2,
    batch_size=BATCH_SIZE,
    learning_rate=1e-3,
    verbose=10,
    seed=SEED,
)

with Timer() as train_time_v2:
    model_v2.fit(data)

print("Took {} seconds for training (model_v2).".format(train_time_v2))

In [ ]:
with Timer() as test_time_v2:
    users, items, preds = [], [], []
    item_list = list(train.itemID.unique())
    for user in train.userID.unique():
        users.extend([user] * len(item_list))
        items.extend(item_list)
        preds.extend(list(model_v2.predict(users[-len(item_list):], items[-len(item_list):], is_list=True)))

    all_predictions_v2 = pd.DataFrame(data={"userID": users, "itemID": items, "prediction": preds})
    merged_v2 = pd.merge(train, all_predictions_v2, on=["userID", "itemID"], how="outer")
    all_predictions_v2 = merged_v2[merged_v2.rating.isnull()].drop('rating', axis=1)

eval_precision_v2 = precision_at_k(test, all_predictions_v2, col_prediction='prediction', k=TOP_K)
eval_recall_v2 = recall_at_k(test, all_predictions_v2, col_prediction='prediction', k=TOP_K)
print("Precision@K (model_v2):\t%f" % eval_precision_v2)
print("Recall@K (model_v2):\t%f" % eval_recall_v2)

### 9. Busca conjunta precisão + diversidade (seleção automática do ponto de equilíbrio)

Aqui adicionamos duas métricas de diversidade que faltavam para poder comparar precisão e diversidade na mesma régua:

- **ILD (Intra-List Diversity)**: `1 - similaridade média entre pares de itens dentro da lista recomendada` de cada usuário, depois calculada a média entre usuários. Quanto maior, mais diversa a lista.
- **Cobertura de catálogo**: fração de itens distintos do catálogo que aparecem em pelo menos uma recomendação, somando todos os usuários.

Fazemos uma grade de λ × γ, calculamos Precision@K, Recall@K, ILD e cobertura para cada ponto, e então **escolhemos automaticamente** a combinação que maximiza o mínimo entre `precisão relativa` e `diversidade relativa` (ambas normalizadas pelo seu próprio máximo na grade) — ou seja, o ponto onde as duas ficam o mais parecidas e altas possível, com desempate a favor de precisão maior quando há empate técnico. Isso é o critério max-min ("fairness") clássico para balancear dois objetivos concorrentes, e responde diretamente ao que você pediu: as duas sobem juntas até onde dá, e se não puderem ficar exatamente iguais, a precisão vence o desempate.

In [ ]:
def intra_list_diversity(reranked_df, sim=None, id_to_col=None):
    """Media (entre usuarios) de 1 - similaridade media par-a-par dentro da lista recomendada.
    sim/id_to_col: passe as versões da validação (seção 10) para medir ILD sem tocar no treino inteiro."""
    sim = item_sim if sim is None else sim
    id_to_col = item_id_to_col if id_to_col is None else id_to_col
    diversities = []
    for _, group in reranked_df.groupby('userID'):
        cols = [id_to_col[i] for i in group['itemID'] if i in id_to_col]
        n = len(cols)
        if n < 2:
            continue
        sim_block = sim[np.ix_(cols, cols)]
        off_diag_mean = (sim_block.sum() - np.trace(sim_block)) / (n * (n - 1))
        diversities.append(1 - off_diag_mean)
    return float(np.mean(diversities)) if diversities else 0.0


def catalog_coverage(reranked_df, n_items_total):
    return reranked_df['itemID'].nunique() / n_items_total

In [ ]:
# Usa o modelo mais forte se a secao 8 foi rodada; senao cai para o modelo original
preds_for_search = (all_predictions_blend if 'all_predictions_blend' in dir()
                    else all_predictions_v2 if 'all_predictions_v2' in dir()
                    else all_predictions)

lambdas = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3]
gammas = [0.0, 0.05, 0.10, 0.15, 0.20]
n_items_total = train['itemID'].nunique()

grid_rows = []
for lam in lambdas:
    for gamma in gammas:
        reranked = build_reranked_predictions(preds_for_search, TOP_K, lam, gamma)
        grid_rows.append({
            'lambda': lam,
            'gamma': gamma,
            'precision@k': precision_at_k(test, reranked, col_prediction='prediction', k=TOP_K),
            'recall@k': recall_at_k(test, reranked, col_prediction='prediction', k=TOP_K),
            'ild': intra_list_diversity(reranked),
            'coverage': catalog_coverage(reranked, n_items_total),
        })

grid_df = pd.DataFrame(grid_rows)
grid_df

In [ ]:
# Selecao automatica do ponto de equilibrio (max-min entre precisao relativa e diversidade relativa)
precision_baseline = grid_df.loc[(grid_df['lambda'] == 1.0) & (grid_df['gamma'] == 0.0), 'precision@k'].iloc[0]
ild_max = grid_df['ild'].max()

grid_df['precision_rel'] = grid_df['precision@k'] / precision_baseline
grid_df['diversity_rel'] = grid_df['ild'] / ild_max
grid_df['balance'] = grid_df[['precision_rel', 'diversity_rel']].min(axis=1)

best = grid_df.sort_values(['balance', 'precision_rel'], ascending=[False, False]).iloc[0]

print("Melhor ponto de equilibrio (precisao e diversidade o mais parecidas e altas possivel):")
print(best)

reranked_best = build_reranked_predictions(preds_for_search, TOP_K, best['lambda'], best['gamma'])
print("\nMetricas finais com lambda={:.2f}, gamma={:.2f}:".format(best['lambda'], best['gamma']))
print("Precision@K:\t%f" % precision_at_k(test, reranked_best, col_prediction='prediction', k=TOP_K))
print("Recall@K:\t%f" % recall_at_k(test, reranked_best, col_prediction='prediction', k=TOP_K))
print("MAP@K:\t\t%f" % map_at_k(test, reranked_best, col_prediction='prediction', k=TOP_K))
print("NDCG@K:\t\t%f" % ndcg_at_k(test, reranked_best, col_prediction='prediction', k=TOP_K))
print("ILD:\t\t%f" % intra_list_diversity(reranked_best))
print("Coverage:\t%f" % catalog_coverage(reranked_best, n_items_total))

**Como ler o resultado**: `precision_rel` e `diversity_rel` são cada métrica dividida pelo seu próprio máximo possível na grade (o máximo de precisão é sempre o baseline `λ=1, γ=0`; o máximo de ILD é o ponto mais diversificado testado). O `balance` é o mínimo dos dois — maximizar isso é o mesmo que buscar o ponto onde nenhuma das duas fica muito atrás da outra. Se, mesmo no ponto ótimo, `diversity_rel` não alcançar `precision_rel`, o desempate (`sort` por `precision_rel` em segundo critério) garante que a escolha final não sacrifica mais precisão do que o necessário — ou seja, em caso de conflito, a precisão vence, como pedido.

Importante: essa seção otimiza o **trade-off**, ela não quebra o trade-off. O teto absoluto de precisão continua sendo definido pela seção 8 (capacidade do modelo). Se ainda estiver bem abaixo de 30–35% depois de tudo isso, o gargalo é o modelo de relevância, não o re-ranqueador — e a única forma de mudar isso é subir `EPOCHS_V2`/capacidade do `model_v2`, ou trocar o protocolo de avaliação, como já discutido na seção 7.


## 10. Validação própria, com múltiplas fatias: escolhendo hiperparâmetros sem tocar no teste

Esta seção corrige o vazamento de dados das seções 4.6, 6 e 9 (que olhavam o teste para ajustar hiperparâmetro) e, a partir desta versão, também reduz o risco de escolher um hiperparâmetro que só parece bom porque encaixou no ruído de UMA fatia de validação (foi o que aconteceu entre as versões anteriores: um resultado bom numa rodada e pior na seguinte, com grades maiores).

O que muda: em vez de uma fatia de validação (85%/15%), construímos **múltiplas fatias cronológicas** (`VAL_SPLIT_FRACS`), cada uma treinando seu(s) próprio(s) NCF(s) de validação do zero. Todo hiperparâmetro (capacidade do NCF, EASE, popularidade recente, MMR) é escolhido pela **média da métrica entre as fatias**, não por uma fatia só. Isso custa mais tempo de treino (cada fatia = um ou dois treinos completos de NCF), mas o valor escolhido tende a generalizar melhor para o teste.

1. Construímos as fatias e treinamos os candidatos de NCF em cada uma (10.1–10.2).
2. Decidimos, pela média entre fatias, se o modelo maior (`model_v2`) realmente compensa (10.2b).
3. Escolhemos `lambda_reg`, meias-vidas, `W_NCF` e popularidade recente pela média entre fatias (10.3).
4. Escolhemos λ, γ e `pool_size` do MMR pela média entre fatias, com piso de precisão (10.4).
5. Só então aplicamos essa configuração já treinada no `train` inteiro e medimos no `test` **uma única vez** (10.5–10.6). Esse é o número que deve ir no TCC.

> **Custo:** com `VAL_SPLIT_FRACS = [0.80, 0.85]` (2 fatias), o tempo de treino de NCF praticamente dobra em relação à versão de fatia única. Para mais robustez (e mais tempo), acrescente uma fração, ex. `0.90`.

**Novo (10.3b):** tentamos subir a precisão do próprio `Rel(u,i)` (recência por posição, T-EASE, RP3-beta, fusão por RRF); cada melhoria só entra se ganhar nas duas fatias de validação, com IC95% pareado acima de zero. Se nenhuma passar, o notebook se comporta exatamente como antes.

**Novo (10.4b):** além do MMR guloso, testamos um re-ranking global por programação linear (GLPK) com teto de exposição por item; o re-ranker final é escolhido pela validação.


### 10.1 Construir as fatias de validação e treinar os candidatos de NCF em cada uma


In [ ]:

VAL_SPLIT_FRACS = [0.80, 0.85]  # cada fração = uma fatia de validação diferente, mesmo corte cronológico por usuário.
# Duas fatias já reduzem bastante o risco de "sorte de fatia única". Para mais robustez, acrescente ex. 0.90
# (custa mais um treino completo de NCF por fração adicionada).

folds = []
for frac in VAL_SPLIT_FRACS:
    train_fit_k, val_k = python_chrono_split(train, frac)
    val_k = val_k[val_k["userID"].isin(train_fit_k["userID"].unique())]
    val_k = val_k[val_k["itemID"].isin(train_fit_k["itemID"].unique())]

    tag = str(frac).replace('.', '')
    train_fit_file_k = f"./train_fit_{tag}.csv"
    val_file_k = f"./val_{tag}.csv"
    train_fit_k.to_csv(train_fit_file_k, index=False)
    val_k.to_csv(val_file_k, index=False)
    data_val_k = NCFDataset(train_file=train_fit_file_k, test_file=val_file_k, seed=SEED)

    model_val_v1_k = NCF(
        n_users=data_val_k.n_users, n_items=data_val_k.n_items, model_type="NeuMF",
        n_factors=4, layer_sizes=[16, 8, 4], n_epochs=EPOCHS, batch_size=BATCH_SIZE,
        learning_rate=1e-3, verbose=10, seed=SEED,
    )
    with Timer() as t_k1:
        model_val_v1_k.fit(data_val_k)
    print(f"[fatia frac={frac}] model_val_v1 treinado em {t_k1} s")

    model_val_v2_k = None
    if 'model_v2' in dir():
        model_val_v2_k = NCF(
            n_users=data_val_k.n_users, n_items=data_val_k.n_items, model_type="NeuMF",
            n_factors=16, layer_sizes=[64, 32, 16, 8], n_epochs=EPOCHS_V2, batch_size=BATCH_SIZE,
            learning_rate=1e-3, verbose=10, seed=SEED,
        )
        with Timer() as t_k2:
            model_val_v2_k.fit(data_val_k)
        print(f"[fatia frac={frac}] model_val_v2 treinado em {t_k2} s")

    folds.append({
        'frac': frac, 'train_fit': train_fit_k, 'val': val_k,
        'model_val_v1': model_val_v1_k, 'model_val_v2': model_val_v2_k,
    })

print(f"\n{len(folds)} fatia(s) de validação construída(s): {[f['frac'] for f in folds]}")


### 10.2 Predições dos NCFs de validação em cada fatia (itens não vistos no respectivo `train_fit`)


In [ ]:

def _predict_full_ranking(model, train_fit_df):
    users, items, preds = [], [], []
    item_list_val = list(train_fit_df.itemID.unique())
    for user in train_fit_df.userID.unique():
        users.extend([user] * len(item_list_val))
        items.extend(item_list_val)
        preds.extend(list(model.predict(users[-len(item_list_val):], items[-len(item_list_val):], is_list=True)))
    preds_df = pd.DataFrame(data={"userID": users, "itemID": items, "prediction": preds})
    merged = pd.merge(train_fit_df, preds_df, on=["userID", "itemID"], how="outer")
    return merged[merged.rating.isnull()].drop('rating', axis=1)

for f in folds:
    with Timer() as t_pred1:
        f['all_predictions_val_v1'] = _predict_full_ranking(f['model_val_v1'], f['train_fit'])
    print(f"[fatia frac={f['frac']}] predição model_val_v1: {t_pred1} s")
    f['all_predictions_val_v2'] = None
    if f['model_val_v2'] is not None:
        with Timer() as t_pred2:
            f['all_predictions_val_v2'] = _predict_full_ranking(f['model_val_v2'], f['train_fit'])
        print(f"[fatia frac={f['frac']}] predição model_val_v2: {t_pred2} s")


### 10.2b O modelo maior (`model_v2`) realmente generaliza melhor, ou só decorou o treino?

Decisão pela **média de Precision@K entre as fatias** — não pela primeira fatia que aparecer. Isso substitui o `'model_v2' in dir()` que era usado antes.


In [ ]:

prec_v1_by_fold, prec_v2_by_fold = [], []
for f in folds:
    p1 = precision_at_k(f['val'], f['all_predictions_val_v1'], col_prediction='prediction', k=TOP_K)
    prec_v1_by_fold.append(p1)
    print(f"[fatia frac={f['frac']}] model_val_v1 -> Precision@K: {p1:.4f}")
    if f['all_predictions_val_v2'] is not None:
        p2 = precision_at_k(f['val'], f['all_predictions_val_v2'], col_prediction='prediction', k=TOP_K)
        prec_v2_by_fold.append(p2)
        print(f"[fatia frac={f['frac']}] model_val_v2 -> Precision@K: {p2:.4f}")

mean_prec_v1 = float(np.mean(prec_v1_by_fold))
USE_MODEL_V2 = False
print(f"\nMédia entre fatias - model_val_v1: {mean_prec_v1:.4f}")
if prec_v2_by_fold:
    mean_prec_v2 = float(np.mean(prec_v2_by_fold))
    print(f"Média entre fatias - model_val_v2: {mean_prec_v2:.4f}")
    if mean_prec_v2 > mean_prec_v1:
        USE_MODEL_V2 = True
        print("\n-> Usando model_v2 dai pra frente: generalizou melhor, em média, nas fatias de validação.")
    else:
        print("\n-> Mantendo o modelo original: model_v2 não superou, em média, nas fatias de validação.")

for f in folds:
    f['all_predictions_val'] = (
        f['all_predictions_val_v2'] if (USE_MODEL_V2 and f['all_predictions_val_v2'] is not None)
        else f['all_predictions_val_v1']
    )


### 10.3 Escolher o `Rel(u,i)`: EASE (λ_reg, meias-vidas), `W_NCF` e popularidade recente — média entre fatias

Mesma lógica de tolerância + desempate conservador de antes, agora aplicada sobre a média entre fatias em vez de uma fatia só. `W_NCF` mantém o piso de 0,1 (o NCF continua parte real do `Rel(u,i)`).


In [ ]:

EASE_LAMBDA_GRID = [10, 25, 50, 100, 200, 300, 500]
HALF_LIFE_GRID = [0, 120, 60, 30]
HALF_LIFE_USER_GRID = [0, 30, 14, 7, 3]
W_NCF_GRID = [0.1, 0.2, 0.3, 0.4]

# ---------- Etapa 1: EASE + W_NCF (média entre fatias) ----------
tuning_rows_val = []
for hl in HALF_LIFE_GRID:
    for hlu in HALF_LIFE_USER_GRID:
        for lam_reg in EASE_LAMBDA_GRID:
            precs_by_fold_w = {w: [] for w in W_NCF_GRID}
            for f in folds:
                scores_i, users_i, items_i = train_ease(
                    f['train_fit'], lambda_reg=lam_reg, half_life_days=hl, user_half_life_days=hlu)
                for w in W_NCF_GRID:
                    blend_i = blend_ncf_ease(f['all_predictions_val'], scores_i, users_i, items_i, w_ncf=w)
                    precs_by_fold_w[w].append(
                        precision_at_k(f['val'], blend_i, col_prediction='prediction', k=TOP_K))
            for w in W_NCF_GRID:
                tuning_rows_val.append({
                    'ease_lambda': lam_reg, 'half_life_days': hl, 'half_life_user': hlu, 'w_ncf': w,
                    'precision@k': float(np.mean(precs_by_fold_w[w])),
                    'precision@k_std': float(np.std(precs_by_fold_w[w])),
                })

tuning_df_val = pd.DataFrame(tuning_rows_val).sort_values('precision@k', ascending=False)
print('Etapa 1 - Top 10 combinações (média entre fatias de validação):')
display(tuning_df_val.head(10))

def _check_edge(nome, valor, grade):
    if valor == min(grade) or valor == max(grade):
        print(f"  ATENCAO: {nome}={valor} caiu na BORDA da grade {grade} -> vale ampliar a grade nessa direcao.")

# Seleção robusta: tolerância + desempate conservador (meias-vidas mais longas), não o argmax puro.
PRECISION_TOL = 0.003
best_prec_etapa1 = tuning_df_val['precision@k'].max()
near_best_etapa1 = tuning_df_val[tuning_df_val['precision@k'] >= best_prec_etapa1 - PRECISION_TOL]
print(f"\n{len(near_best_etapa1)} combinação(ões) dentro de {PRECISION_TOL} do topo ({best_prec_etapa1:.4f}); "
      f"escolhendo a mais conservadora entre elas.")
best_row_val = near_best_etapa1.sort_values(
    ['half_life_user', 'half_life_days'], ascending=[False, False]
).iloc[0]
BEST_EASE_LAMBDA = float(best_row_val['ease_lambda'])
BEST_HALF_LIFE = float(best_row_val['half_life_days'])
BEST_HALF_LIFE_USER = float(best_row_val['half_life_user'])
BEST_W_NCF = float(best_row_val['w_ncf'])
print(f"\nEtapa 1 escolheu: lambda_reg={BEST_EASE_LAMBDA}, half_life={BEST_HALF_LIFE}, "
      f"half_life_user={BEST_HALF_LIFE_USER}, W_NCF={BEST_W_NCF}")
_check_edge('lambda_reg', BEST_EASE_LAMBDA, EASE_LAMBDA_GRID)
_check_edge('half_life', BEST_HALF_LIFE, HALF_LIFE_GRID)
_check_edge('half_life_user', BEST_HALF_LIFE_USER, HALF_LIFE_USER_GRID)
_check_edge('W_NCF', BEST_W_NCF, W_NCF_GRID)

prec_sem_pop = float(best_row_val['precision@k'])
for f in folds:
    f['ease_scores_val'], f['ease_users_val'], f['ease_items_val'] = train_ease(
        f['train_fit'], lambda_reg=BEST_EASE_LAMBDA, half_life_days=BEST_HALF_LIFE,
        user_half_life_days=BEST_HALF_LIFE_USER)

# ---------- Etapa 2: popularidade recente como terceiro termo (média entre fatias) ----------
W_POP_GRID = [0.05, 0.10, 0.15, 0.20, 0.30, 0.40]
POP_WINDOW_GRID = [7, 14, 30, 60]
pop_rows = []
for win in POP_WINDOW_GRID:
    for wp in W_POP_GRID:
        precs = []
        for f in folds:
            pop_map_i = make_recent_popularity(f['train_fit'], window_days=win)
            blend_i = blend_ncf_ease(f['all_predictions_val'], f['ease_scores_val'], f['ease_users_val'],
                                     f['ease_items_val'], w_ncf=BEST_W_NCF, pop_map=pop_map_i, w_pop=wp)
            precs.append(precision_at_k(f['val'], blend_i, col_prediction='prediction', k=TOP_K))
        pop_rows.append({'pop_window': win, 'w_pop': wp, 'precision@k': float(np.mean(precs))})

pop_df = pd.DataFrame(pop_rows).sort_values('precision@k', ascending=False)
print('\nEtapa 2 - popularidade recente (média entre fatias; referência sem popularidade: %f):' % prec_sem_pop)
display(pop_df.head(5))

BEST_W_POP, BEST_POP_WINDOW = 0.0, 30
best_prec_pop = pop_df['precision@k'].max()
if best_prec_pop > prec_sem_pop + 0.004:
    near_best_pop = pop_df[pop_df['precision@k'] >= best_prec_pop - PRECISION_TOL]
    chosen_pop = near_best_pop.sort_values('pop_window', ascending=False).iloc[0]
    BEST_W_POP = float(chosen_pop['w_pop'])
    BEST_POP_WINDOW = int(chosen_pop['pop_window'])
    print(f"Popularidade recente ENTRA: w_pop={BEST_W_POP}, janela={BEST_POP_WINDOW} dias")
    _check_edge('w_pop', BEST_W_POP, W_POP_GRID)
    _check_edge('pop_window', BEST_POP_WINDOW, POP_WINDOW_GRID)
else:
    print("Popularidade recente NÃO entra (ganho médio <= 0.004 nas fatias de validação).")

for f in folds:
    pop_map_val_f = make_recent_popularity(f['train_fit'], window_days=BEST_POP_WINDOW) if BEST_W_POP > 0 else None
    f['all_predictions_val_blend'] = blend_ncf_ease(
        f['all_predictions_val'], f['ease_scores_val'], f['ease_users_val'], f['ease_items_val'],
        w_ncf=BEST_W_NCF, pop_map=pop_map_val_f, w_pop=BEST_W_POP)


### 10.3b Subindo a precisão: melhorias candidatas para o `Rel(u,i)` (cada uma só entra se ganhar na validação)

**Por que aqui.** O re-ranking (MMR ou ILP/GLPK) só redistribui a precisão que o `Rel(u,i)` já trouxe; ele não cria precisão nova. O teto, portanto, é decidido na 10.3. E a própria saída da 10.3 aponta caminhos concretos: (i) a Precision@K de validação **sobe a cada vez que a meia-vida do usuário diminui** (30 → 7 → 3 dias) e o melhor ficou na borda da grade; (ii) o `W_NCF` ficou no piso de 0,1, isto é, o EASE faz quase todo o trabalho de acurácia; (iii) a regra de desempate "mais conservadora" da 10.3 abriu mão de ~0,003 de Precision@K de validação.

**O que esta seção testa**, sempre sobre as mesmas fatias de validação e sem tocar no teste:

| Etapa | Ideia | Por que poderia ajudar |
|---|---|---|
| A | Recência do histórico do usuário: meias-vidas menores (até 0,5 dia) **e** decaimento por **posição** (as últimas N interações pesam mais) | O teste é "o que o usuário consome logo depois"; no ml-100k muita gente avalia tudo em poucos dias, e a posição distingue o que o calendário não distingue |
| B | **T-EASE**: EASE com alvo temporal (aprende a prever a cauda recente do histórico a partir da cabeça) | Treina com a mesma tarefa do teste (prever o futuro), em vez de reconstruir o próprio passado |
| C | **RP3-beta** (passeio aleatório com penalização de popularidade) | Modelo de vizinhança que a literatura de reprodutibilidade (Dacrema et al., 2019) mostra competitivo com modelos neurais; erra diferente do EASE, então o *ensemble* tende a ganhar |
| D | Fusão por **RRF** (1/(k+posição)) em vez de rank percentual | Dá mais peso ao topo da lista de cada modelo, que é onde a Precision@10 é decidida |

**Travas contra "ganho que é só sorte da grade"** (todas obrigatórias para uma melhoria entrar): (1) ganho médio de Precision@K ≥ `ADOPT_MIN_GAIN` (0,003); (2) a melhoria vence em **todas** as fatias; (3) o IC95% (bootstrap pareado por usuário) do ganho fica acima de zero. Antes de tudo, há uma **conferência**: o avaliador rápido usado aqui precisa reproduzir o `precision_at_k` oficial; se não reproduzir, nada é adotado. Se nada passar nas travas, **nada muda** no resto do notebook.

**O NCF continua parte real do `Rel(u,i)`** (peso `W_NCF` ≥ 0,1, como antes), e MMR + cosseno + NicheScore (ou o ILP/GLPK) seguem aplicados por cima, sem alteração. Se algum membro novo for adotado, a 10.4 já parte do `Rel(u,i)` novo e a 10.5 o aplica no treino inteiro com a mesma configuração (nada é re-otimizado no treino inteiro).

> **Custo:** poucos minutos (não treina nenhum NCF novo; reaproveita as predições de validação da 10.2).

In [ ]:
# ---- 10.3b-1: peças (membros do Rel, avaliador rápido, fusão) ----
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, diags
from scipy.stats import rankdata

def _codes(train_df):
    u_cat = train_df['userID'].astype('category')
    i_cat = train_df['itemID'].astype('category')
    return (u_cat.cat.codes.values, i_cat.cat.codes.values,
            list(u_cat.cat.categories), list(i_cat.cat.categories))

def _pos_from_last(train_df, ucodes):
    """Posição de cada interação contada a partir da MAIS RECENTE do próprio usuário (0 = a última)."""
    tmp = pd.DataFrame({'u': ucodes, 'ts': train_df['timestamp'].values.astype(float),
                        'k': np.arange(len(train_df))}).sort_values(['u', 'ts', 'k'])
    out = np.empty(len(train_df), dtype=float)
    out[tmp['k'].values] = tmp.groupby('u').cumcount(ascending=False).values
    return out

def _weights(train_df, ucodes, hl_days=None, hl_pos=None):
    """Peso de cada interação: decaimento por IDADE (dias, mesma fórmula do train_ease) e/ou por POSIÇÃO
    (nº de interações do usuário desde a mais recente). Sem argumentos => tudo pesa 1."""
    w = np.ones(len(train_df))
    if hl_days:
        ts = train_df['timestamp'].values.astype(float)
        w = w * 0.5 ** (((ts.max() - ts) / 86400.0) / float(hl_days))
    if hl_pos:
        w = w * 0.5 ** (_pos_from_last(train_df, ucodes) / float(hl_pos))
    return w

def _sparse(ucodes, icodes, vals, shape):
    return csr_matrix((vals, (ucodes, icodes)), shape=shape).astype(np.float64)

def user_matrix(train_df, hl_days=None, hl_pos=None):
    """Vetor de histórico de cada usuário (com pesos de recência) usado na hora de pontuar: X_user @ B."""
    uc, ic, users, items = _codes(train_df)
    return _sparse(uc, ic, _weights(train_df, uc, hl_days, hl_pos), (len(users), len(items)))

def ease_B(train_df, lambda_reg, hl_days=None):
    """Matriz B do EASE (idêntica à do train_ease; aqui separada do passo de pontuar, para reaproveitar o B)."""
    uc, ic, users, items = _codes(train_df)
    X = _sparse(uc, ic, _weights(train_df, uc, hl_days=hl_days), (len(users), len(items)))
    G = (X.T @ X).toarray()
    d = np.diag_indices(G.shape[0])
    G[d] += lambda_reg
    P = np.linalg.inv(G)
    B = P / (-np.diag(P))
    B[d] = 0.0
    return B

def temporal_ridge_B(train_df, lambda_reg, tail_frac):
    """EASE com ALVO TEMPORAL ("T-EASE"): em vez de reconstruir o próprio histórico (EASE), aprende a prever a
    CAUDA recente de cada usuário (últimos `tail_frac` do histórico dele no treino) a partir da CABEÇA (o resto):
        B = argmin ||X_cauda - X_cabeça B||^2 + lambda ||B||^2  =  (X_cabeçaᵀ X_cabeça + λI)^-1 X_cabeçaᵀ X_cauda
    Isso imita, dentro do treino, a tarefa do teste (prever o que o usuário consome DEPOIS). Cabeça e cauda não
    compartilham itens, então não precisa zerar a diagonal. Só usa timestamps do próprio treino."""
    uc, ic, users, items = _codes(train_df)
    shape = (len(users), len(items))
    n_u = np.bincount(uc, minlength=len(users))
    tail_n = np.clip(np.round(tail_frac * n_u).astype(int), 1, np.maximum(n_u - 1, 1))
    is_tail = _pos_from_last(train_df, uc) < tail_n[uc]
    Xh = _sparse(uc[~is_tail], ic[~is_tail], np.ones(int((~is_tail).sum())), shape)
    Xt = _sparse(uc[is_tail], ic[is_tail], np.ones(int(is_tail.sum())), shape)
    G = (Xh.T @ Xh).toarray()
    G[np.diag_indices(G.shape[0])] += lambda_reg
    return np.linalg.solve(G, (Xh.T @ Xt).toarray())

def rp3beta_S(train_df, beta):
    """RP3-beta (Paudel et al., 2016): passeio aleatório usuário->item->usuário->item com penalização de
    popularidade: S_ij = (P_iu P_ui)_ij / pop_j^beta, diagonal zerada. beta=0 favorece populares; beta alto, nicho."""
    uc, ic, users, items = _codes(train_df)
    X = _sparse(uc, ic, np.ones(len(train_df)), (len(users), len(items)))
    d_u = np.asarray(X.sum(axis=1)).ravel()
    p_i = np.asarray(X.sum(axis=0)).ravel()
    Pui = diags(1.0 / d_u) @ X          # usuário -> item
    Piu = diags(1.0 / p_i) @ X.T        # item -> usuário
    S = (Piu @ Pui).toarray() / (p_i[None, :] ** beta)
    np.fill_diagonal(S, 0.0)
    return S

def _pct_rows(M, cand):
    """Rank percentual [0,1] por linha, SÓ entre os candidatos (itens não vistos) — igual ao percentile_normalize
    aplicado por usuário no blend_ncf_ease. Fora dos candidatos fica 0."""
    out = np.zeros(M.shape, dtype=float)
    for r in range(M.shape[0]):
        c = cand[r]
        n = int(c.sum())
        if n > 1:
            out[r, c] = (rankdata(M[r, c], method='average') - 1) / (n - 1)
    return out

def _fuse(pct, cand, fusion='percentile', rrf_k=60):
    """'percentile' = o blend atual. 'rrf' = Reciprocal Rank Fusion 1/(k + posição): valoriza mais o TOPO da lista
    de cada modelo e menos a cauda (onde os modelos discordam por ruído)."""
    if fusion == 'percentile':
        return pct
    n = cand.sum(axis=1, keepdims=True)
    rank_desc = n - pct * (n - 1)      # 1 = melhor candidato
    return np.where(cand, 1.0 / (rrf_k + rank_desc), 0.0)


class RelFold:
    """Tudo o que o Rel(u,i) precisa para UMA partição (fatia de validação ou treino inteiro), em matrizes densas
    usuário x item: candidatos (itens não vistos), rank percentual do NCF e (se houver `val_df`) os acertos."""
    def __init__(self, train_df, ncf_df, val_df=None):
        self.train_df = train_df
        _, _, self.users, self.items = _codes(train_df)
        nU, nI = len(self.users), len(self.items)
        self.uc = pd.Categorical(ncf_df['userID'], categories=self.users).codes
        self.ic = pd.Categorical(ncf_df['itemID'], categories=self.items).codes
        ok = (self.uc >= 0) & (self.ic >= 0)
        self.ok = ok
        NCF = np.full((nU, nI), np.nan)
        NCF[self.uc[ok], self.ic[ok]] = ncf_df['prediction'].values[ok]
        self.cand = ~np.isnan(NCF)
        self.pct_ncf = _pct_rows(np.nan_to_num(NCF), self.cand)
        self._B, self._X, self._ease_hl_cache = {}, {}, {}
        self.VAL, self.n_val_users = None, None
        if val_df is not None:
            vu = pd.Categorical(val_df['userID'], categories=self.users).codes
            vi = pd.Categorical(val_df['itemID'], categories=self.items).codes
            vok = (vu >= 0) & (vi >= 0)
            self.VAL = np.zeros((nU, nI), dtype=bool)
            self.VAL[vu[vok], vi[vok]] = True
            self.n_val_users = val_df['userID'].nunique()

    # --- caches de B e de X_user (o B é caro; o X_user é barato) ---
    def B(self, kind, **kw):
        key = (kind,) + tuple(sorted(kw.items()))
        if key not in self._B:
            if kind != 'ease':   # T-EASE/RP3 ocupam ~20 MB cada: guarda só o mais recente de cada tipo
                for k_ in [k_ for k_ in self._B if k_[0] == kind]:
                    del self._B[k_]
            if kind == 'ease':
                self._B[key] = ease_B(self.train_df, kw['lam'], kw.get('hl') or None)
            elif kind == 'tease':
                self._B[key] = temporal_ridge_B(self.train_df, kw['lam'], kw['tail'])
            elif kind == 'rp3':
                self._B[key] = rp3beta_S(self.train_df, kw['beta'])
        return self._B[key]

    def X_user(self, uhl, upos):
        key = (uhl or None, upos or None)
        if key not in self._X:
            self._X[key] = user_matrix(self.train_df, hl_days=key[0], hl_pos=key[1])
        return self._X[key]

    def member_pct(self, kind, uhl, upos, **kw):
        scores = np.asarray(self.X_user(uhl, upos) @ self.B(kind, **kw))
        return _pct_rows(scores, self.cand)

    def pop_pct(self, window_days):
        pop_map = make_recent_popularity(self.train_df, window_days=window_days)
        vec = np.array([pop_map.get(it, 0) for it in self.items], dtype=float)
        return _pct_rows(np.tile(vec, (len(self.users), 1)), self.cand)

    def precision_users(self, S, k):
        """Precision@K de cada usuário (vetor); só os usuários que têm itens na validação (os demais ficam de fora)."""
        S2 = np.where(self.cand, S, -np.inf)
        idx = np.argpartition(-S2, k - 1, axis=1)[:, :k]
        hits = np.take_along_axis(self.VAL, idx, axis=1).sum(axis=1)
        return (hits / k)[self.VAL.any(axis=1)]

    def precision(self, S, k):
        S2 = np.where(self.cand, S, -np.inf)
        idx = np.argpartition(-S2, k - 1, axis=1)[:, :k]
        hits = np.take_along_axis(self.VAL, idx, axis=1).sum(axis=1)
        return float(hits.sum() / k / self.n_val_users)

    def to_df(self, ncf_df, S):
        out = ncf_df.copy()
        vals = S[np.clip(self.uc, 0, None), np.clip(self.ic, 0, None)]
        out['prediction'] = np.where(self.ok, vals, -1e9)
        return out


def rel_weights(cfg):
    """Pesos do Rel(u,i): NCF e popularidade têm peso próprio; o resto (M) é dividido entre EASE, T-EASE e RP3-beta."""
    M = 1.0 - cfg['w_ncf'] - cfg['w_pop']
    s_t = cfg['tease']['share'] if cfg.get('tease') else 0.0
    s_r = cfg['rp3']['share'] if cfg.get('rp3') else 0.0
    return {'ncf': cfg['w_ncf'], 'pop': cfg['w_pop'],
            'ease': M * (1 - s_t - s_r), 'tease': M * s_t, 'rp3': M * s_r}

def rel_scores(fold, cfg, pcts=None):
    """Matriz de scores Rel(u,i) (usuário x item) para a configuração `cfg`.
    pcts (opcional): dict {'ease'|'tease'|'rp3'|'pop': matriz de rank percentual já calculada}, só para
    não recalcular o mesmo membro dezenas de vezes durante a busca em grade."""
    pcts = pcts or {}
    w = rel_weights(cfg)
    uhl, upos = cfg['user_hl_days'], cfg['user_hl_pos']
    fz, rk = cfg['fusion'], cfg.get('rrf_k') or 60
    S = w['ncf'] * _fuse(fold.pct_ncf, fold.cand, fz, rk)
    if w['ease'] > 0:
        p = pcts['ease'] if 'ease' in pcts else fold.member_pct(
            'ease', uhl, upos, lam=cfg['ease_lambda'], hl=cfg['ease_hl'])
        S = S + w['ease'] * _fuse(p, fold.cand, fz, rk)
    if w['tease'] > 0:
        p = pcts['tease'] if 'tease' in pcts else fold.member_pct(
            'tease', uhl, upos, lam=cfg['tease']['lam'], tail=cfg['tease']['tail'])
        S = S + w['tease'] * _fuse(p, fold.cand, fz, rk)
    if w['rp3'] > 0:
        p = pcts['rp3'] if 'rp3' in pcts else fold.member_pct('rp3', uhl, upos, beta=cfg['rp3']['beta'])
        S = S + w['rp3'] * _fuse(p, fold.cand, fz, rk)
    if w['pop'] > 0:
        p = pcts['pop'] if 'pop' in pcts else fold.pop_pct(cfg['pop_window'])
        S = S + w['pop'] * _fuse(p, fold.cand, fz, rk)
    return S

def rel_blend_df(train_df, ncf_df, cfg):
    """Versão em dataframe (userID, itemID, prediction) do Rel(u,i) novo, pronta para o MMR/ILP.
    Usada no treino inteiro (seção 10.5) quando a configuração da 10.3b foi adotada."""
    fold = RelFold(train_df, ncf_df)
    return fold.to_df(ncf_df, rel_scores(fold, cfg))

In [ ]:
# ---- 10.3b-2: montar as partições, conferir o avaliador rápido e fixar a linha de base ----
ADOPT_MIN_GAIN = 0.003   # ganho mínimo de Precision@K (média entre fatias) para uma melhoria ENTRAR

rel_folds = [RelFold(f['train_fit'], f['all_predictions_val'], f['val']) for f in folds]

# Linha de base = exatamente o Rel(u,i) que a 10.3 escolheu (EASE + NCF [+ popularidade, se entrou])
BASE_CFG = dict(
    ease_lambda=BEST_EASE_LAMBDA, ease_hl=BEST_HALF_LIFE,
    user_hl_days=BEST_HALF_LIFE_USER, user_hl_pos=0,
    w_ncf=BEST_W_NCF, w_pop=BEST_W_POP, pop_window=BEST_POP_WINDOW,
    tease=None, rp3=None, fusion='percentile', rrf_k=None,
)

def _fold_precs(cfg, pcts_per_fold=None):
    return [rf.precision(rel_scores(rf, cfg, None if pcts_per_fold is None else pcts_per_fold[i]), TOP_K)
            for i, rf in enumerate(rel_folds)]

base_precs = _fold_precs(BASE_CFG)

# Conferência: o avaliador rápido (matrizes) tem que dar o MESMO Precision@K que a função oficial do recommenders
ref_precs = [precision_at_k(f['val'], f['all_predictions_val_blend'], col_prediction='prediction', k=TOP_K)
             for f in folds]
for f, a, b in zip(folds, base_precs, ref_precs):
    print(f"[fatia frac={f['frac']}] Precision@K  avaliador rápido: {a:.4f} | precision_at_k oficial: {b:.4f}")
ADOPT_ENABLED = max(abs(a - b) for a, b in zip(base_precs, ref_precs)) < 0.002
if ADOPT_ENABLED:
    print("Conferência OK: pode-se confiar no avaliador rápido.")
else:
    print("ATENÇÃO: o avaliador rápido NÃO bate com o oficial -> nada será adotado nesta seção (a 10.4 segue com o blend atual).")
print(f"Linha de base (média entre fatias): {np.mean(base_precs):.4f}")

CUR_CFG = dict(BASE_CFG)
CUR_PRECS = list(base_precs)

def _stage_pcts(cfg):
    """Rank percentual dos membros JÁ ativos em `cfg`, uma vez por fatia (reaproveitado em toda a grade)."""
    out = []
    for rf in rel_folds:
        d = {'ease': rf.member_pct('ease', cfg['user_hl_days'], cfg['user_hl_pos'],
                                   lam=cfg['ease_lambda'], hl=cfg['ease_hl'])}
        if cfg['w_pop'] > 0:
            d['pop'] = rf.pop_pct(cfg['pop_window'])
        if cfg.get('tease'):
            d['tease'] = rf.member_pct('tease', cfg['user_hl_days'], cfg['user_hl_pos'],
                                       lam=cfg['tease']['lam'], tail=cfg['tease']['tail'])
        if cfg.get('rp3'):
            d['rp3'] = rf.member_pct('rp3', cfg['user_hl_days'], cfg['user_hl_pos'], beta=cfg['rp3']['beta'])
        out.append(d)
    return out

def _paired_ci(cfg_new, n_boot=2000, seed=0):
    """IC95% (bootstrap pareado por usuário, fatias empilhadas) do ganho de Precision@K de `cfg_new` sobre CUR_CFG.
    É uma trava de segurança contra "ganho" que é só sorte da grade; não é um teste de hipótese formal
    (os mesmos usuários aparecem nas duas fatias)."""
    d = np.concatenate([rf.precision_users(rel_scores(rf, cfg_new), TOP_K) - rf.precision_users(rel_scores(rf, CUR_CFG), TOP_K)
                        for rf in rel_folds])
    rng = np.random.default_rng(seed)
    means = d[rng.integers(0, len(d), size=(n_boot, len(d)))].mean(axis=1)
    return tuple(np.percentile(means, [2.5, 97.5]))

def _decide(rows, label, cfg_of_row):
    """Escolhe o melhor da grade e decide se ENTRA. Três travas, todas obrigatórias:
       (1) ganho médio >= ADOPT_MIN_GAIN; (2) vitória em TODAS as fatias; (3) IC95% pareado por usuário acima de 0."""
    df = pd.DataFrame(rows).sort_values('precision@k', ascending=False).reset_index(drop=True)
    fold_cols = [c for c in df.columns if c.startswith('fatia_')]
    best = df.iloc[0]
    gain = best['precision@k'] - np.mean(CUR_PRECS)
    wins = all(best[c] > b for c, b in zip(fold_cols, CUR_PRECS))
    lo, hi = _paired_ci(cfg_of_row(best))
    adopt = bool(ADOPT_ENABLED and gain >= ADOPT_MIN_GAIN and wins and lo > 0)
    print(f"\n{label} - top 5 da grade (média entre fatias; referência atual: {np.mean(CUR_PRECS):.4f}):")
    display(df.head(5).round(4))
    print(f"Melhor: ganho médio {gain:+.4f} | vence em todas as fatias: {wins} | "
          f"IC95% do ganho pareado por usuário: [{lo:+.4f}, {hi:+.4f}]  ->  "
          + ("ENTRA." if adopt else "NÃO entra (fica como estava)."))
    return best, adopt

In [ ]:
# ---- 10.3b-3: recência do HISTÓRICO do usuário (por dias e por posição) ----
# A própria 10.3 mostrou que Precision@K sobe quando a meia-vida do usuário diminui (30d -> 7d -> 3d) e que a
# grade ficou na borda. Aqui estendemos para menos dias e adicionamos o decaimento por POSIÇÃO (as últimas N
# interações do usuário pesam mais), que também funciona para quem avaliou tudo no mesmo dia.
UHL_GRID = [0, 30, 14, 7, 3, 2, 1, 0.5]    # meia-vida em dias (0 = desligado)
UPOS_GRID = [0, 80, 40, 20, 10, 5]         # meia-vida em nº de interações (0 = desligado)

rows = []
for uhl in UHL_GRID:
    for upos in UPOS_GRID:
        precs = _fold_precs(dict(CUR_CFG, user_hl_days=uhl, user_hl_pos=upos))
        row = {'user_hl_days': uhl, 'user_hl_pos': upos, 'precision@k': float(np.mean(precs))}
        row.update({f'fatia_{i}': p for i, p in enumerate(precs)})
        rows.append(row)

best, adopt = _decide(rows, 'Etapa A - recência do usuário',
                      lambda r: dict(CUR_CFG, user_hl_days=r['user_hl_days'], user_hl_pos=r['user_hl_pos']))
if adopt:
    CUR_CFG.update(user_hl_days=float(best['user_hl_days']), user_hl_pos=float(best['user_hl_pos']))
    CUR_PRECS = [float(best[f'fatia_{i}']) for i in range(len(rel_folds))]
    print(f"-> user_hl_days={CUR_CFG['user_hl_days']}, user_hl_pos={CUR_CFG['user_hl_pos']}")
    if best['user_hl_days'] in (min(g for g in UHL_GRID if g > 0),) or best['user_hl_pos'] in (min(g for g in UPOS_GRID if g > 0),):
        print("  ATENCAO: o melhor caiu na BORDA (menor valor não-zero) da grade -> vale ampliar nessa direção.")

In [ ]:
# ---- 10.3b-4: T-EASE (EASE com alvo temporal) como segundo membro do Rel ----
# O teste é "o que o usuário consome DEPOIS do corte". O EASE normal aprende a reconstruir o próprio histórico;
# o T-EASE aprende, dentro do treino, a prever a CAUDA recente de cada usuário a partir da CABEÇA (ver a função).
TEASE_LAMBDAS = [50, 200, 500]
TEASE_TAILS = [0.1, 0.2, 0.3]
TEASE_SHARES = [0.2, 0.4, 0.6, 0.8]   # fração do peso não-NCF que vai para o T-EASE (o resto fica com o EASE)

pcts_cur = _stage_pcts(CUR_CFG)
rows = []
for lam in TEASE_LAMBDAS:
    for tail in TEASE_TAILS:
        tease_pct = [rf.member_pct('tease', CUR_CFG['user_hl_days'], CUR_CFG['user_hl_pos'], lam=lam, tail=tail)
                     for rf in rel_folds]
        for share in TEASE_SHARES:
            cfg = dict(CUR_CFG, tease=dict(lam=lam, tail=tail, share=share))
            precs = _fold_precs(cfg, [dict(pcts_cur[i], tease=tease_pct[i]) for i in range(len(rel_folds))])
            row = {'tease_lambda': lam, 'tail_frac': tail, 'share': share, 'precision@k': float(np.mean(precs))}
            row.update({f'fatia_{i}': p for i, p in enumerate(precs)})
            rows.append(row)
        del tease_pct

best, adopt = _decide(rows, 'Etapa B - T-EASE',
                      lambda r: dict(CUR_CFG, tease=dict(lam=r['tease_lambda'], tail=r['tail_frac'], share=r['share'])))
if adopt:
    CUR_CFG['tease'] = dict(lam=float(best['tease_lambda']), tail=float(best['tail_frac']), share=float(best['share']))
    CUR_PRECS = [float(best[f'fatia_{i}']) for i in range(len(rel_folds))]
    print(f"-> T-EASE entra: {CUR_CFG['tease']}")

In [ ]:
# ---- 10.3b-5: RP3-beta (passeio aleatório com penalização de popularidade) como terceiro membro ----
RP3_BETAS = [0.2, 0.4, 0.6, 0.8]
RP3_SHARES = [0.15, 0.30, 0.50]

pcts_cur = _stage_pcts(CUR_CFG)
s_t = CUR_CFG['tease']['share'] if CUR_CFG.get('tease') else 0.0
rows = []
for beta in RP3_BETAS:
    rp3_pct = [rf.member_pct('rp3', CUR_CFG['user_hl_days'], CUR_CFG['user_hl_pos'], beta=beta) for rf in rel_folds]
    for share in RP3_SHARES:
        if s_t + share >= 1.0:
            continue
        cfg = dict(CUR_CFG, rp3=dict(beta=beta, share=share))
        precs = _fold_precs(cfg, [dict(pcts_cur[i], rp3=rp3_pct[i]) for i in range(len(rel_folds))])
        row = {'rp3_beta': beta, 'share': share, 'precision@k': float(np.mean(precs))}
        row.update({f'fatia_{i}': p for i, p in enumerate(precs)})
        rows.append(row)
    del rp3_pct

best, adopt = _decide(rows, 'Etapa C - RP3-beta',
                      lambda r: dict(CUR_CFG, rp3=dict(beta=r['rp3_beta'], share=r['share'])))
if adopt:
    CUR_CFG['rp3'] = dict(beta=float(best['rp3_beta']), share=float(best['share']))
    CUR_PRECS = [float(best[f'fatia_{i}']) for i in range(len(rel_folds))]
    print(f"-> RP3-beta entra: {CUR_CFG['rp3']}")

In [ ]:
# ---- 10.3b-6: como juntar os modelos: rank percentual (atual) x Reciprocal Rank Fusion ----
# O rank percentual dá o mesmo peso à cauda e ao topo da lista de cada modelo; o RRF (1/(k+posição)) dá mais peso
# ao topo, que é onde o Precision@10 é decidido.
RRF_KS = [10, 30, 60, 120]

pcts_cur = _stage_pcts(CUR_CFG)
rows = []
for rk in RRF_KS:
    cfg = dict(CUR_CFG, fusion='rrf', rrf_k=rk)
    precs = _fold_precs(cfg, pcts_cur)
    row = {'fusion': 'rrf', 'rrf_k': rk, 'precision@k': float(np.mean(precs))}
    row.update({f'fatia_{i}': p for i, p in enumerate(precs)})
    rows.append(row)

best, adopt = _decide(rows, 'Etapa D - fusão por RRF',
                      lambda r: dict(CUR_CFG, fusion='rrf', rrf_k=int(r['rrf_k'])))
if adopt:
    CUR_CFG.update(fusion='rrf', rrf_k=int(best['rrf_k']))
    CUR_PRECS = [float(best[f'fatia_{i}']) for i in range(len(rel_folds))]
    print(f"-> fusão RRF entra: k={CUR_CFG['rrf_k']}")

In [ ]:
# ---- 10.3b-7: fechar a configuração do Rel(u,i) e repassá-la para a 10.4 ----
REL_CFG = dict(CUR_CFG)
REL_CFG_IS_BASELINE = (REL_CFG == BASE_CFG)

print("Configuração do Rel(u,i) escolhida na validação:")
for k_, v_ in REL_CFG.items():
    print(f"  {k_}: {v_}")
print(f"\nPrecision@K de validação (média entre fatias): linha de base {np.mean(base_precs):.4f} -> "
      f"final {np.mean(CUR_PRECS):.4f}  ({np.mean(CUR_PRECS) - np.mean(base_precs):+.4f})")
print("Por fatia: base", [round(p, 4) for p in base_precs], "-> final", [round(p, 4) for p in CUR_PRECS])

if REL_CFG_IS_BASELINE:
    print("\nNada superou a linha de base com margem e consistência: a 10.4 segue com o blend atual (nada muda).")
else:
    for f, rf in zip(folds, rel_folds):
        f['all_predictions_val_blend'] = rf.to_df(f['all_predictions_val'], rel_scores(rf, REL_CFG))
    print("\nRel(u,i) novo adotado: as fatias de validação foram atualizadas e a 10.4 (MMR/ILP) já parte dele.")
    print("A 10.5 aplicará a MESMA configuração no treino inteiro.")

### 10.4 Escolher λ, γ e tamanho do pool do MMR — média entre fatias, com piso de precisão


In [ ]:

for f in folds:
    f['item_sim_val'], f['item_id_to_col_val'] = compute_item_similarity(f['train_fit'])
    f['niche_score_val'] = make_niche_score(f['train_fit'])
    f['n_items_val'] = f['train_fit']['itemID'].nunique()

lambdas_search = [1.0, 0.99, 0.98, 0.95, 0.9]
gammas_search = [0.0, 0.10, 0.20, 0.30, 0.40, 0.50]
pools_search = [15, 20, 30, 50]

grid_rows_val = []
for pool in pools_search:
    for lam in lambdas_search:
        for gamma in gammas_search:
            precs, ilds, covs = [], [], []
            for f in folds:
                reranked = build_reranked_predictions(
                    f['all_predictions_val_blend'], TOP_K, lam, gamma, pool_size=pool,
                    sim=f['item_sim_val'], id_to_col=f['item_id_to_col_val'], niche_fn=f['niche_score_val']
                )
                precs.append(precision_at_k(f['val'], reranked, col_prediction='prediction', k=TOP_K))
                ilds.append(intra_list_diversity(reranked, sim=f['item_sim_val'], id_to_col=f['item_id_to_col_val']))
                covs.append(catalog_coverage(reranked, f['n_items_val']))
            grid_rows_val.append({
                'pool': pool, 'lambda': lam, 'gamma': gamma,
                'precision@k': float(np.mean(precs)), 'ild': float(np.mean(ilds)), 'coverage': float(np.mean(covs)),
            })

grid_df_val = pd.DataFrame(grid_rows_val)
precision_baseline_val = grid_df_val.loc[
    (grid_df_val['lambda'] == 1.0) & (grid_df_val['gamma'] == 0.0) & (grid_df_val['pool'] == 50), 'precision@k'
].iloc[0]
print("Precision@K de referência (média entre fatias; λ=1, γ=0, pool=50): %f" % precision_baseline_val)

grid_df_val['precision_rel'] = grid_df_val['precision@k'] / precision_baseline_val
grid_df_val['diversity_rel'] = 0.5 * (grid_df_val['ild'] / grid_df_val['ild'].max()
                                      + grid_df_val['coverage'] / grid_df_val['coverage'].max())
grid_df_val['balance'] = grid_df_val[['precision_rel', 'diversity_rel']].min(axis=1)

PRECISION_FLOOR = 0.98
candidates = grid_df_val[grid_df_val['precision_rel'] >= PRECISION_FLOOR]
if len(candidates) > 0:
    DIVERSITY_TOL = 0.01
    best_div = candidates['diversity_rel'].max()
    near_best_cands = candidates[candidates['diversity_rel'] >= best_div - DIVERSITY_TOL]
    best_mmr_val = near_best_cands.sort_values(['lambda', 'pool'], ascending=[False, True]).iloc[0]
    print(f"Melhor ponto com piso de precisão >= {PRECISION_FLOOR:.0%} da referência "
          f"(entre {len(near_best_cands)} empatados em diversidade, o menos agressivo):")
else:
    best_mmr_val = grid_df_val.sort_values(['balance', 'precision_rel'], ascending=[False, False]).iloc[0]
    print(f"Nenhuma combinação manteve >= {PRECISION_FLOOR:.0%}; usando o critério de equilíbrio (min):")

BEST_LAMBDA_MMR = float(best_mmr_val['lambda'])
BEST_GAMMA_MMR = float(best_mmr_val['gamma'])
BEST_POOL_SIZE = int(best_mmr_val['pool'])
print(best_mmr_val)
_check_edge('lambda (MMR)', BEST_LAMBDA_MMR, lambdas_search)
_check_edge('gamma (MMR)', BEST_GAMMA_MMR, gammas_search)
_check_edge('pool_size', BEST_POOL_SIZE, pools_search)


### 10.4b Re-ranking GLOBAL por programação linear (GLPK): teto de exposição por item

O MMR (10.4) decide usuário por usuário e nunca controla quantas listas cada filme ocupa. Aqui formulamos o re-ranking **para todos os usuários de uma vez**:

- variável `x_ui` (usuário u recebe o item i do seu pool);
- cada usuário recebe exatamente K itens: `Σ_i x_ui = K`;
- **teto de exposição:** cada item aparece em no máximo `C = ceil(α · nº de usuários)` listas;
- objetivo: maximizar `Σ (rel_ui + γ · niche_ui) · x_ui` (mesma relevância percentílica e mesmo NicheScore do MMR).

A matriz de restrições é a de um grafo bipartido (totalmente unimodular): o LP já tem solução inteira, sem branch-and-bound — o GLPK resolve por simplex. Os hiperparâmetros (α, γ, pool) são escolhidos na validação (média entre fatias) com o **mesmo piso de precisão** do MMR; depois comparamos os dois re-rankers também pela validação e usamos o melhor.

> **Instalação do GLPK (Windows).** O `pip install pulp` (dentro do `.venv311`) instala só a *interface*; o resolvedor é o programa `glpsol.exe`, que vem à parte: baixe o **WinGLPK** (zip no SourceForge), extraia numa pasta sem espaços (ex.: `C:\glpk`) e aponte `GLPSOL_PATH` (na célula abaixo) para o `glpsol.exe` da subpasta `w64` — ou coloque essa pasta no PATH do Windows e reinicie o VS Code. No conda: `conda install -c conda-forge glpk`. **Confira a primeira linha impressa pela célula abaixo: tem que ser `Solver do ILP global: glpk`.** Sem isso, o código usa o HiGHS (via scipy) como reserva — resolve o **mesmo** problema, mas os números *não* devem ser citados como "GLPK" no TCC.
>
> **Tempo:** a busca em grade (10.4b) resolve dezenas de LPs. Se o GLPK demorar demais nela, troque `ILP_GRID_SOLVER` para `'highs'` na célula da grade (o ótimo do LP é o mesmo) e deixe o GLPK para o resultado final (10.5).

In [ ]:

import math
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix

# Se o glpsol NÃO estiver no PATH (comum no Windows), aponte aqui para o executável.
# Exemplo: GLPSOL_PATH = r"C:\glpk\w64\glpsol.exe"
GLPSOL_PATH = None

def _glpk_cmd(**kw):
    import pulp
    return pulp.GLPK_CMD(path=GLPSOL_PATH, **kw)

def _glpk_available():
    try:
        return bool(_glpk_cmd(msg=False).available())
    except Exception:
        return False

ILP_SOLVER = 'glpk' if _glpk_available() else 'highs'
print(f"Solver do ILP global: {ILP_SOLVER}" + ("" if ILP_SOLVER == 'glpk' else
      "  (GLPK/pulp não encontrado -> usando HiGHS como reserva; instale `pip install pulp` + o glpsol e/ou preencha GLPSOL_PATH acima, para usar o GLPK)"))


def _solve_assignment_lp(var_user, var_item, var_w, n_users, n_items, k, cap, solver):
    """max sum(w*x)  s.a.  sum_i x_ui = k (cada usuário), sum_u x_ui <= cap (cada item), 0 <= x <= 1.
    A matriz de restrições é a de incidência de um grafo bipartido (totalmente unimodular), então o ótimo
    do LP já é inteiro: não precisa de branch-and-bound. Levanta RuntimeError se for inviável."""
    N = len(var_w)
    if solver == 'glpk':
        import pulp
        prob = pulp.LpProblem("global_rerank", pulp.LpMaximize)
        x = [pulp.LpVariable(f"x{v}", lowBound=0, upBound=1) for v in range(N)]
        prob += pulp.lpSum(float(var_w[v]) * x[v] for v in range(N))
        by_user = [[] for _ in range(n_users)]
        by_item = [[] for _ in range(n_items)]
        for v in range(N):
            by_user[var_user[v]].append(v)
            by_item[var_item[v]].append(v)
        for u in range(n_users):
            prob += pulp.lpSum(x[v] for v in by_user[u]) == k
        for it in range(n_items):
            prob += pulp.lpSum(x[v] for v in by_item[it]) <= cap
        prob.solve(_glpk_cmd(msg=False, options=["--tmlim", "600"]))
        if pulp.LpStatus[prob.status] != 'Optimal':
            raise RuntimeError(f"GLPK status: {pulp.LpStatus[prob.status]}")
        return np.array([(x[v].value() or 0.0) for v in range(N)])

    from scipy.optimize import linprog
    A_eq = coo_matrix((np.ones(N), (var_user, np.arange(N))), shape=(n_users, N)).tocsr()
    A_ub = coo_matrix((np.ones(N), (var_item, np.arange(N))), shape=(n_items, N)).tocsr()
    res = linprog(-np.asarray(var_w, dtype=float), A_ub=A_ub, b_ub=np.full(n_items, float(cap)),
                  A_eq=A_eq, b_eq=np.full(n_users, float(k)), bounds=(0, 1), method='highs')
    if not res.success:
        raise RuntimeError(f"HiGHS: {res.message}")
    return res.x


def global_ilp_rerank(preds_df, k, gamma, alpha, pool_size=30, niche_fn=None, solver=None):
    """Re-ranking GLOBAL (todos os usuários de uma vez) por programação linear/inteira.
    Maximiza  sum_{u,i} (rel_ui + gamma * niche_ui) * x_ui   sujeito a
        - cada usuário recebe exatamente k itens do seu pool (top `pool_size` do blend);
        - cada item aparece em no máximo C = ceil(alpha * n_usuarios) listas (teto de exposição).
    alpha=None => sem teto (equivale a top-k puro, com gamma=0).
    Devolve dataframe userID/itemID/prediction no mesmo formato de build_reranked_predictions."""
    niche_fn = niche_score if niche_fn is None else niche_fn
    solver = ILP_SOLVER if solver is None else solver

    users, item_ids = [], {}
    var_user, var_item, var_w, var_rel = [], [], [], []
    for u_idx, (user_id, group) in enumerate(preds_df.groupby('userID')):
        pool = group.nlargest(min(pool_size, len(group)), 'prediction')
        items = pool['itemID'].tolist()
        rel = percentile_normalize(pool['prediction'].values)
        niche = percentile_normalize(np.array([niche_fn(user_id, i) for i in items], dtype=float))
        users.append(user_id)
        for it, r, nn in zip(items, rel, niche):
            j = item_ids.setdefault(it, len(item_ids))
            var_user.append(u_idx); var_item.append(j)
            var_w.append(r + gamma * nn); var_rel.append(r)

    var_user = np.asarray(var_user); var_item = np.asarray(var_item)
    var_w = np.asarray(var_w, dtype=float); var_rel = np.asarray(var_rel, dtype=float)
    n_users, n_items = len(users), len(item_ids)
    inv_items = {j: it for it, j in item_ids.items()}

    cap = n_users if alpha is None else max(math.ceil(alpha * n_users), math.ceil(n_users * k / n_items))
    x = None
    for _ in range(6):  # se o teto for inviável para os pools existentes, afrouxa 25% e tenta de novo
        try:
            x = _solve_assignment_lp(var_user, var_item, var_w, n_users, n_items, k, cap, solver)
            break
        except RuntimeError:
            cap = math.ceil(cap * 1.25)
    if x is None:
        raise RuntimeError("ILP global inviável mesmo após afrouxar o teto de exposição.")

    starts = np.concatenate([[0], np.cumsum(np.bincount(var_user, minlength=n_users))])
    rows = []
    for u_idx in range(n_users):
        s, e = starts[u_idx], starts[u_idx + 1]
        xs, rs = x[s:e], var_rel[s:e]
        top = np.lexsort((-rs, -xs))[:k]            # maiores x (desempate por relevância)
        top = top[np.argsort(-rs[top], kind='stable')]  # lista final ordenada por relevância
        for rank, loc in enumerate(top):
            rows.append((users[u_idx], inv_items[var_item[s + loc]], k - rank))
    return pd.DataFrame(rows, columns=['userID', 'itemID', 'prediction'])


In [ ]:

ILP_GRID_SOLVER = ILP_SOLVER   # troque para 'highs' se o GLPK demorar demais na busca em grade (o ótimo do LP é o mesmo)
ILP_ALPHA_GRID = [0.08, 0.12, 0.20, 0.30, 0.50]   # fração dos usuários que pode ver o mesmo item (teto de exposição)
ILP_GAMMA_GRID = [0.0, 0.10, 0.20, 0.30]
ILP_POOL_GRID = [20, 30]

ilp_rows = []
for pool in ILP_POOL_GRID:
    for alpha in ILP_ALPHA_GRID:
        for gamma in ILP_GAMMA_GRID:
            precs, ilds, covs = [], [], []
            for f in folds:
                rr = global_ilp_rerank(f['all_predictions_val_blend'], TOP_K, gamma, alpha,
                                       pool_size=pool, niche_fn=f['niche_score_val'], solver=ILP_GRID_SOLVER)
                precs.append(precision_at_k(f['val'], rr, col_prediction='prediction', k=TOP_K))
                ilds.append(intra_list_diversity(rr, sim=f['item_sim_val'], id_to_col=f['item_id_to_col_val']))
                covs.append(catalog_coverage(rr, f['n_items_val']))
            ilp_rows.append({'pool': pool, 'alpha': alpha, 'gamma': gamma,
                             'precision@k': float(np.mean(precs)), 'ild': float(np.mean(ilds)),
                             'coverage': float(np.mean(covs))})

grid_ilp_val = pd.DataFrame(ilp_rows)
grid_ilp_val['precision_rel'] = grid_ilp_val['precision@k'] / precision_baseline_val

# Normalização COMUM aos dois re-rankers (só para compará-los; não altera a escolha do MMR na 10.4)
ild_ref = max(grid_df_val['ild'].max(), grid_ilp_val['ild'].max())
cov_ref = max(grid_df_val['coverage'].max(), grid_ilp_val['coverage'].max())
def _div_cmp(ild, cov):
    return 0.5 * (ild / ild_ref + cov / cov_ref)
grid_ilp_val['div_cmp'] = _div_cmp(grid_ilp_val['ild'], grid_ilp_val['coverage'])

print("Fronteira precisão x diversidade do ILP global (validação, média entre fatias):")
display(grid_ilp_val.sort_values('precision_rel', ascending=False).head(15).round(4))

ILP_PRECISION_FLOOR = PRECISION_FLOOR
ILP_DIV_TOL = 0.01
cands_ilp = grid_ilp_val[grid_ilp_val['precision_rel'] >= ILP_PRECISION_FLOOR]
if len(cands_ilp) > 0:
    near_ilp = cands_ilp[cands_ilp['div_cmp'] >= cands_ilp['div_cmp'].max() - ILP_DIV_TOL]
    best_ilp_val = near_ilp.sort_values(['alpha', 'pool'], ascending=[False, True]).iloc[0]  # o menos agressivo
    ilp_meets_floor = True
    print(f"\nMelhor ponto do ILP respeitando o piso de {ILP_PRECISION_FLOOR:.0%}:")
else:
    best_ilp_val = grid_ilp_val.sort_values('precision_rel', ascending=False).iloc[0]
    ilp_meets_floor = False
    print(f"\nNenhum ponto do ILP manteve >= {ILP_PRECISION_FLOOR:.0%}; mostrando o de maior precisão:")
print(best_ilp_val)

BEST_ILP_ALPHA = float(best_ilp_val['alpha'])
BEST_ILP_GAMMA = float(best_ilp_val['gamma'])
BEST_ILP_POOL = int(best_ilp_val['pool'])
_check_edge('alpha (ILP)', BEST_ILP_ALPHA, ILP_ALPHA_GRID)
_check_edge('gamma (ILP)', BEST_ILP_GAMMA, ILP_GAMMA_GRID)
_check_edge('pool (ILP)', BEST_ILP_POOL, ILP_POOL_GRID)

# Escolha do re-ranker: pela validação. O ILP só substitui o MMR se respeitar o piso E ganhar em diversidade
# por margem (parcimônia: 0.005), senão mantemos o MMR guloso.
div_mmr = _div_cmp(best_mmr_val['ild'], best_mmr_val['coverage'])
mmr_meets_floor = bool(best_mmr_val['precision_rel'] >= PRECISION_FLOOR)
div_ilp = float(best_ilp_val['div_cmp'])
RERANKER = 'mmr'
if ilp_meets_floor and (not mmr_meets_floor or div_ilp > div_mmr + 0.005):
    RERANKER = 'ilp'
print(f"\nDiversidade (comum) na validação -> MMR: {div_mmr:.4f} | ILP: {div_ilp:.4f}")
print(f"Re-ranker escolhido para o resultado final: {RERANKER.upper()}")


### 10.5 Aplicar a configuração escolhida no treino inteiro (sem re-otimizar nada)

In [ ]:

# 10.3b é opcional: se não foi executada, tudo segue como antes
REL_CFG_IS_BASELINE = globals().get('REL_CFG_IS_BASELINE', True)

# NOVO: usa o modelo maior so se ele tiver vencido honestamente na validacao (secao 10.2b);
# nao usa so porque a secao 8 foi executada.
base_preds_for_blend = all_predictions_v2 if USE_MODEL_V2 else all_predictions

if REL_CFG_IS_BASELINE:
    # Reajusta o EASE no TREINO INTEIRO (não em train_fit) com o lambda_reg escolhido na validação
    ease_scores_final, ease_users_final, ease_items_final = train_ease(train, lambda_reg=BEST_EASE_LAMBDA, half_life_days=BEST_HALF_LIFE,
                                                                            user_half_life_days=BEST_HALF_LIFE_USER)
    pop_map_final = make_recent_popularity(train, window_days=BEST_POP_WINDOW) if BEST_W_POP > 0 else None
    all_predictions_blend_final = blend_ncf_ease(
        base_preds_for_blend, ease_scores_final, ease_users_final, ease_items_final,
        w_ncf=BEST_W_NCF, pop_map=pop_map_final, w_pop=BEST_W_POP
    )
else:
    # A 10.3b adotou um Rel(u,i) novo: aplica a MESMA configuração (escolhida na validação) no TREINO INTEIRO
    all_predictions_blend_final = rel_blend_df(train, base_preds_for_blend, REL_CFG)
    print("Rel(u,i) da 10.3b aplicado no treino inteiro:", REL_CFG)

# Re-ranking com o item_sim / niche_score "oficiais" (treino inteiro) e os hiperparâmetros escolhidos na validação
reranked_mmr_final = build_reranked_predictions(
    all_predictions_blend_final, TOP_K, BEST_LAMBDA_MMR, BEST_GAMMA_MMR, pool_size=BEST_POOL_SIZE
)  # sim/id_to_col/niche_fn=None -> usa item_sim/item_id_to_col/niche_score globais (treino inteiro)
reranked_ilp_final = global_ilp_rerank(
    all_predictions_blend_final, TOP_K, BEST_ILP_GAMMA, BEST_ILP_ALPHA, pool_size=BEST_ILP_POOL
)  # niche_fn=None -> niche_score global (treino inteiro)
reranked_final = reranked_ilp_final if RERANKER == 'ilp' else reranked_mmr_final

print(f"Configuração final: USE_MODEL_V2={USE_MODEL_V2}, lambda_reg={BEST_EASE_LAMBDA}, half_life={BEST_HALF_LIFE}, half_life_user={BEST_HALF_LIFE_USER}, w_pop={BEST_W_POP}, pool={BEST_POOL_SIZE}, W_NCF={BEST_W_NCF}, "
      f"lambda={BEST_LAMBDA_MMR}, gamma={BEST_GAMMA_MMR} | ILP: alpha={BEST_ILP_ALPHA}, gamma={BEST_ILP_GAMMA}, pool={BEST_ILP_POOL} | RERANKER={RERANKER}")


### 10.6 Resultado final — este é o número para reportar no TCC (teste tocado uma única vez)

In [ ]:

n_items_total = train['itemID'].nunique()

print("=== Resultado final no TESTE (hiperparâmetros escolhidos só na validação) ===")
print("Precision@K:\t%f" % precision_at_k(test, reranked_final, col_prediction='prediction', k=TOP_K))
print("Recall@K:\t%f" % recall_at_k(test, reranked_final, col_prediction='prediction', k=TOP_K))
print("MAP@K:\t\t%f" % map_at_k(test, reranked_final, col_prediction='prediction', k=TOP_K))
print("NDCG@K:\t\t%f" % ndcg_at_k(test, reranked_final, col_prediction='prediction', k=TOP_K))
print("ILD:\t\t%f" % intra_list_diversity(reranked_final))
print("Coverage:\t%f" % catalog_coverage(reranked_final, n_items_total))


### 10.7 Tabela do MVP: NCF puro × blend sem diversificar × re-rankers (MMR guloso e ILP global/GLPK)

**Hipótese do trabalho (critério do MVP):** a diversificação não compromete significativamente a acurácia. Critério: Precision@10 do modelo final **acima do NCF puro**, com ganho relevante de ILD e cobertura.

A tabela é só de **relato** — os dois re-rankers aparecem lado a lado, mas **quem vira o 'modelo final' foi decidido na validação** (10.4b), não olhando esta tabela. A linha "blend sem diversificar" isola quanto a diversificação custa em precisão.


In [ ]:

def _metrics_row(name, preds_df):
    return {
        'modelo': name,
        'Precision@10': precision_at_k(test, preds_df, col_prediction='prediction', k=TOP_K),
        'Recall@10': recall_at_k(test, preds_df, col_prediction='prediction', k=TOP_K),
        'MAP@10': map_at_k(test, preds_df, col_prediction='prediction', k=TOP_K),
        'NDCG@10': ndcg_at_k(test, preds_df, col_prediction='prediction', k=TOP_K),
        'ILD': intra_list_diversity(preds_df),
        'Cobertura': catalog_coverage(preds_df, n_items_total),
    }

# NCF puro: top-10 pela ordem do NCF (λ=1, γ=0 sobre as predições do NCF)
ncf_puro_top = build_reranked_predictions(all_predictions, TOP_K, 1.0, 0.0)
# Blend (NCF+EASE) sem diversificar: mesma configuração final, só com λ=1, γ=0
blend_sem_div_top = build_reranked_predictions(all_predictions_blend_final, TOP_K, 1.0, 0.0)

_cfg_rel = globals().get('REL_CFG', {}) or {}
NOME_BLEND = 'Blend ' + '+'.join(['NCF', 'EASE'] + (['T-EASE'] if _cfg_rel.get('tease') else [])
                                  + (['RP3b'] if _cfg_rel.get('rp3') else [])) + ' (sem diversificar)'
NOME_MMR = 'Blend + MMR guloso + NicheScore'
NOME_ILP = 'Blend + ILP global (' + ('GLPK' if ILP_SOLVER == 'glpk' else 'HiGHS') + ') + NicheScore'
NOME_FINAL = NOME_ILP if RERANKER == 'ilp' else NOME_MMR

mvp_df = pd.DataFrame([
    _metrics_row('NCF puro', ncf_puro_top),
    _metrics_row(NOME_BLEND, blend_sem_div_top),
    _metrics_row(NOME_MMR, reranked_mmr_final),
    _metrics_row(NOME_ILP, reranked_ilp_final),
]).set_index('modelo')
display(mvp_df.round(4))
print(f"\nRe-ranker escolhido na VALIDAÇÃO (usado como 'modelo final'): {NOME_FINAL}")

base = mvp_df.loc['NCF puro']
final = mvp_df.loc[NOME_FINAL]
sem = mvp_df.loc[NOME_BLEND]
print("\n--- Modelo final vs NCF puro ---")
for col in mvp_df.columns:
    print(f"{col:14s} {base[col]:.4f} -> {final[col]:.4f}  ({(final[col]/base[col]-1)*100:+.1f}%)")
print("\n--- Custo da diversificação (final vs blend sem diversificar) ---")
for col in ['Precision@10', 'ILD', 'Cobertura']:
    print(f"{col:14s} {sem[col]:.4f} -> {final[col]:.4f}  ({(final[col]/sem[col]-1)*100:+.1f}%)")
print("\nMVP atingido (Precision@10 final > NCF puro)?", bool(final['Precision@10'] > base['Precision@10']))


### 10.8 Barras de erro: as diferenças da tabela são maiores que o ruído?

Com ~900 usuários no teste, uma Precision@10 em torno de 0,20 tem erro-padrão da ordem de 0,6 ponto percentual (IC95% ≈ ±1 p.p.). Diferenças de décimos de ponto entre linhas da 10.7 podem ser só ruído de amostragem de usuários. Esta célula calcula, **sobre o mesmo teste e sem retreinar nada**, a Precision@10 por usuário, o IC95% de cada linha e o IC95% das **diferenças pareadas** (mesmos usuários nos dois lados) — é o que a banca costuma perguntar ("essa diferença é significativa?").

In [ ]:
def per_user_precision(true_df, preds_df, k=TOP_K):
    """Precision@K de cada usuário do teste (mesma definição do precision_at_k: acertos no top-k / k)."""
    top = (preds_df.sort_values(['userID', 'prediction'], ascending=[True, False])
                   .groupby('userID').head(k)[['userID', 'itemID']])
    truth = true_df[['userID', 'itemID']].drop_duplicates().assign(_hit=1)
    hits = top.merge(truth, on=['userID', 'itemID'], how='inner').groupby('userID')['_hit'].sum()
    return hits.reindex(pd.Index(true_df['userID'].unique())).fillna(0.0) / k

def _boot_means(d, n_boot=5000, seed=0):
    rng = np.random.default_rng(seed)
    return d[rng.integers(0, len(d), size=(n_boot, len(d)))].mean(axis=1)

lists_ci = {
    'NCF puro': ncf_puro_top,
    NOME_BLEND: blend_sem_div_top,
    NOME_MMR: reranked_mmr_final,
    NOME_ILP: reranked_ilp_final,
}
pu = {name: per_user_precision(test, df_) for name, df_ in lists_ci.items()}

print("Precision@10 por modelo, com IC95% (bootstrap sobre usuários do teste):")
for name, s in pu.items():
    lo, hi = np.percentile(_boot_means(s.values), [2.5, 97.5])
    print(f"  {name:45s} {s.mean():.4f}  [{lo:.4f}, {hi:.4f}]   (confere com a tabela 10.7: {mvp_df.loc[name, 'Precision@10']:.4f})")

print("\nDiferenças PAREADAS de Precision@10 (mesmos usuários), IC95%:")
pairs = [(NOME_MMR, 'NCF puro'), (NOME_ILP, 'NCF puro'), (NOME_BLEND, 'NCF puro'),
         (NOME_MMR, NOME_BLEND), (NOME_ILP, NOME_BLEND), (NOME_ILP, NOME_MMR)]
for a, b in pairs:
    d = (pu[a] - pu[b]).values
    lo, hi = np.percentile(_boot_means(d), [2.5, 97.5])
    verdict = "diferença distinguível de zero" if (lo > 0 or hi < 0) else "dentro do ruído (IC inclui 0)"
    print(f"  {a:45s} - {b:35s} {d.mean():+.4f}  [{lo:+.4f}, {hi:+.4f}]  -> {verdict}")

print(f"\nSolver usado pelo ILP global nos números acima: {ILP_SOLVER.upper()}"
      + ("" if ILP_SOLVER == 'glpk' else "   <- ATENÇÃO: não é o GLPK! Configure o GLPK (ver 10.4b) antes de citar 'GLPK' no TCC."))


**Como citar isso no TCC:** "Os hiperparâmetros do modelo de relevância (λ_reg e meias-vidas do decaimento temporal do EASE, W_NCF, e o peso da popularidade recente, quando incluída) e dos re-rankers (λ, γ e tamanho do pool do MMR guloso; e γ, tamanho do pool e teto de exposição α do re-ranking global por programação linear resolvido com o GLPK), sujeitos a um piso mínimo de precisão relativa ao blend sem diversificação, foram selecionados por busca em grade, pela média de múltiplas partições de validação cronológicas do conjunto de treino, sem acesso ao conjunto de teste, que foi usado uma única vez para as métricas finais."

> Se a 10.3b adotou alguma melhoria, acrescente à frase acima os itens adotados (impressos na 10.3b): decaimento da recência do histórico (dias e/ou posição), o T-EASE (EASE com alvo temporal), o RP3-beta e/ou a fusão por RRF, todos escolhidos nas mesmas fatias de validação, com o ganho exigido de pelo menos 0,003 de Precision@K, vitória em todas as fatias e IC95% pareado acima de zero. Se nenhuma foi adotada, vale registrar isso também: foram testadas e não superaram o `Rel(u,i)` da 10.3.
